# SLR_09 — Uji kandidat melalui loader v0.2

**Tujuan:** memastikan dua arsip kandidat yang sudah diekspor dapat dimuat bersama menggunakan loader kandidat v0.2, adapter 0.4 dan cutoff 0,95. Tidak melatih ulang, tidak memilih checkpoint baru dan tidak mengulang seluruh evaluasi kualitas.

## Bukti uji sebelumnya tetap berlaku

| Bukti yang sudah ada | Cakupan |
| --- | --- |
| Smoke baseline CPU, 12 narasi | SENA v1 dan NARA v1; berhasil. |
| Reload SENA fine-tune, tiga contoh | Manifest ekspor menyatakan logits cocok setelah reload. |
| Reload dan inferensi NARA fine-tune | Manifest dan run state menyatakan berhasil di Colab. |
| Suite revisi v0.2, 60/60 | Kontrak/fixture sintetis; bukan pemuatan bobot neural. |

Notebook ini menutup celah antara bukti ekspor model dan **jalur pemuatan baru setelah perbaikan**. Kode berasal byte-identik dari paket `SLR09_Paket_Perbaikan_v0.2.zip`, SHA-256 `4d004d3e3f74cb8d698c67ee5571f03653b568e52455e28d025d1ff3dd78e2d6`.

**Cara menjalankan di Google Colab:** unggah notebook, jalankan berurutan, izinkan akses Drive saat diminta. Dua ZIP model sudah berada di `MyDrive/SILAR_SLR09/`. Tidak perlu mengunggah ulang bobot atau mengeksekusi notebook training.

Colab diberi scope **REPRESENTATIVE**. Kelulusan di Colab tidak dinyatakan sebagai pengujian VM tujuan atau persetujuan mutu kelompok. Tidak ada hasil eksekusi baru yang diisikan terlebih dahulu pada notebook ini.


In [ ]:
# 1. Siapkan snapshot kode revisi dan direktori kerja baru.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys
import uuid

EMBEDDED_FILES = json.loads('{"models/slr09/code/active_nara_adapter/legacy_postprocessing.py": "\\"\\"\\"Helper murni dari SILAR_Fusion_Engine.ipynb yang telah diaudit.\\n\\nIsi fungsi/konstanta disalin tanpa perubahan. Hanya impor/globals disesuaikan.\\nTidak menyertakan Fusion lama, uncertainty lama, atau Completeness Engine.\\nLihat legacy_source_trace.json untuk hash asal setiap fungsi.\\n\\"\\"\\"\\nimport re\\nimport numpy as np\\nfrom silar_ai_adapter import NARA_LABELS, TOKEN_PATTERN\\n\\nEXPECTED_SRL = set(NARA_LABELS)\\n\\n\\ndef tokenize_with_offsets(text):\\n    result = []\\n\\n    for match in TOKEN_PATTERN.finditer(text):\\n        result.append({\\n            \\"token\\": match.group(0).lower(),\\n            \\"original\\": match.group(0),\\n            \\"start\\": match.start(),\\n            \\"end\\": match.end(),\\n        })\\n\\n    return result\\n\\n\\ndef contiguous_role_spans(pred_tokens, role):\\n    spans = []\\n    current = []\\n\\n    def flush():\\n        nonlocal current\\n\\n        if current:\\n            spans.append({\\n                \\"role\\": role,\\n                \\"text\\": \\" \\".join(\\n                    x[\\"original\\"]\\n                    for x in current\\n                ),\\n                \\"mean_confidence\\": float(\\n                    np.mean([\\n                        x[\\"confidence\\"]\\n                        for x in current\\n                    ])\\n                ),\\n            })\\n\\n        current = []\\n\\n    for item in pred_tokens:\\n        if item[\\"label\\"] == role:\\n            current.append(item)\\n        else:\\n            flush()\\n\\n    flush()\\n    return spans\\n\\n\\ndef all_srl_spans(pred_tokens):\\n    roles = sorted(EXPECTED_SRL - {\\"O\\"})\\n\\n    return {\\n        role: contiguous_role_spans(\\n            pred_tokens,\\n            role,\\n        )\\n        for role in roles\\n    }\\n\\n\\nTIME_PATTERNS = [\\n    r\\"\\\\b(?:pukul|jam)\\\\s+\\\\d{1,2}(?:[.:]\\\\d{2})?(?:\\\\s*(?:pagi|siang|sore|malam))?\\\\b\\",\\n    r\\"\\\\b(?:tadi|sejak)\\\\s+(?:pagi|siang|sore|malam)\\\\b\\",\\n    r\\"\\\\b(?:pagi|siang|sore|malam)\\\\s+ini\\\\b\\",\\n    r\\"\\\\bmalam\\\\s+tadi\\\\b\\",\\n    r\\"\\\\bkemarin\\\\b\\",\\n    r\\"\\\\bbarusan\\\\b\\",\\n    r\\"\\\\btadi\\\\b\\",\\n    r\\"\\\\bhari\\\\s+ini\\\\b\\",\\n]\\n\\n\\ndef time_fallback(text):\\n    matches = []\\n    occupied = []\\n\\n    for pattern in TIME_PATTERNS:\\n        for m in re.finditer(\\n            pattern,\\n            text,\\n            flags=re.IGNORECASE,\\n        ):\\n            start, end = m.span()\\n\\n            overlaps = any(\\n                start < e and end > s\\n                for s, e in occupied\\n            )\\n\\n            if overlaps:\\n                continue\\n\\n            matches.append({\\n                \\"role\\": \\"TIME-FALLBACK\\",\\n                \\"text\\": m.group(0),\\n                \\"source\\": \\"DETERMINISTIC_REGEX\\",\\n            })\\n\\n            occupied.append((start, end))\\n\\n    return sorted(\\n        matches,\\n        key=lambda x: text.lower().find(\\n            x[\\"text\\"].lower()\\n        ),\\n    )\\n\\n\\nDEATH_NEGATION_PATTERNS = [\\n    r\\"\\\\btidak\\\\s+ada\\\\s+(?:korban\\\\s+)?(?:meninggal|korban\\\\s+jiwa)\\\\b\\",\\n    r\\"\\\\btidak\\\\s+terdapat\\\\s+(?:korban\\\\s+)?(?:meninggal|korban\\\\s+jiwa)\\\\b\\",\\n    r\\"\\\\bnihil\\\\s+(?:korban\\\\s+)?(?:meninggal|korban\\\\s+jiwa)\\\\b\\",\\n    r\\"\\\\btanpa\\\\s+(?:korban\\\\s+)?(?:meninggal|korban\\\\s+jiwa)\\\\b\\",\\n]\\n\\n\\nDEATH_EVIDENCE_PATTERNS = [\\n    r\\"\\\\bmeninggal\\\\b\\",\\n    r\\"\\\\btewas\\\\b\\",\\n    r\\"\\\\bwafat\\\\b\\",\\n    r\\"\\\\bmenewaskan\\\\b\\",\\n    r\\"\\\\bkorban\\\\s+jiwa\\\\b\\",\\n]\\n\\n\\nWOUND_EVIDENCE_PATTERNS = [\\n    r\\"\\\\bterluka\\\\b\\",\\n    r\\"\\\\bluka\\\\b\\",\\n    r\\"\\\\bluka\\\\s+bakar\\\\b\\",\\n    r\\"\\\\bcedera\\\\b\\",\\n    r\\"\\\\bmengalami\\\\s+luka\\\\b\\",\\n    r\\"\\\\bmengeluh\\\\s+sakit\\\\b\\",\\n]\\n\\n\\ndef has_any_pattern(text, patterns):\\n    return any(\\n        re.search(pattern, text, flags=re.IGNORECASE)\\n        for pattern in patterns\\n    )\\n\\n\\ndef has_death_negation(text):\\n    return has_any_pattern(text, DEATH_NEGATION_PATTERNS)\\n\\n\\ndef normalize_victim_phrase(text, kind):\\n    number_word = (\\n        r\\"(?:\\\\d+|satu|dua|tiga|empat|lima|enam|tujuh|\\"\\n        r\\"delapan|sembilan|sepuluh)\\"\\n    )\\n    person_word = r\\"(?:orang|korban|pengendara|penumpang|warga)\\"\\n\\n    if kind == \\"wound\\":\\n        cue = (\\n            r\\"(?:terluka|cedera|luka\\\\s+bakar|\\"\\n            r\\"mengalami\\\\s+luka|mengeluh\\\\s+sakit)\\"\\n        )\\n    else:\\n        cue = r\\"(?:meninggal|tewas|wafat)\\"\\n\\n    pattern = (\\n        rf\\"\\\\b(?:{number_word}\\\\s+)?\\"\\n        rf\\"(?:{person_word}\\\\s+)?\\"\\n        rf\\"{cue}\\\\b\\"\\n    )\\n\\n    match = re.search(pattern, text, flags=re.IGNORECASE)\\n    return match.group(0) if match else None\\n\\n\\ndef apply_victim_safety(text, spans):\\n    safety_flags = []\\n\\n    death_spans = spans.get(\\"DEATHVICTIM-ARG\\", [])\\n    wound_spans = spans.get(\\"WOUNDVICTIM-ARG\\", [])\\n\\n    death_negated = has_death_negation(text)\\n    death_evidence = has_any_pattern(\\n        text,\\n        DEATH_EVIDENCE_PATTERNS,\\n    )\\n    wound_evidence = has_any_pattern(\\n        text,\\n        WOUND_EVIDENCE_PATTERNS,\\n    )\\n\\n    if death_spans and death_negated:\\n        spans[\\"DEATHVICTIM-ARG\\"] = []\\n        safety_flags.append(\\n            \\"DEATH_VICTIM_SUPPRESSED_BY_NEGATION\\"\\n        )\\n\\n    elif death_spans and not death_evidence:\\n        spans[\\"DEATHVICTIM-ARG\\"] = []\\n        safety_flags.append(\\n            \\"DEATH_VICTIM_SUPPRESSED_NO_DEATH_EVIDENCE\\"\\n        )\\n\\n    elif death_spans and death_evidence:\\n        phrase = normalize_victim_phrase(text, \\"death\\")\\n        if phrase:\\n            spans[\\"DEATHVICTIM-ARG\\"] = [{\\n                \\"role\\": \\"DEATHVICTIM-ARG\\",\\n                \\"text\\": phrase,\\n                \\"mean_confidence\\": float(np.mean([\\n                    x.get(\\"mean_confidence\\", 0.0)\\n                    for x in death_spans\\n                ])),\\n                \\"source\\": \\"SRL_PLUS_LEXICAL_EVIDENCE\\",\\n            }]\\n\\n    if wound_spans and not wound_evidence:\\n        spans[\\"WOUNDVICTIM-ARG\\"] = []\\n        safety_flags.append(\\n            \\"WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE\\"\\n        )\\n\\n    elif wound_spans and wound_evidence:\\n        phrase = normalize_victim_phrase(text, \\"wound\\")\\n        if phrase:\\n            spans[\\"WOUNDVICTIM-ARG\\"] = [{\\n                \\"role\\": \\"WOUNDVICTIM-ARG\\",\\n                \\"text\\": phrase,\\n                \\"mean_confidence\\": float(np.mean([\\n                    x.get(\\"mean_confidence\\", 0.0)\\n                    for x in wound_spans\\n                ])),\\n                \\"source\\": \\"SRL_PLUS_LEXICAL_EVIDENCE\\",\\n            }]\\n\\n    return spans, safety_flags\\n\\n\\ndef normalize_display_text(text):\\n    text = str(text)\\n\\n    text = re.sub(\\n        r\\"\\\\s+([,;:!?])\\",\\n        r\\"\\\\1\\",\\n        text,\\n    )\\n\\n    text = re.sub(\\n        r\\"(?<=\\\\d)\\\\s*\\\\.\\\\s*(?=\\\\d)\\",\\n        \\".\\",\\n        text,\\n    )\\n\\n    text = re.sub(\\n        r\\"\\\\s+\\\\.\\",\\n        \\".\\",\\n        text,\\n    )\\n\\n    text = re.sub(\\n        r\\"\\\\s{2,}\\",\\n        \\" \\",\\n        text,\\n    )\\n\\n    return text.strip()\\n\\n\\ndef span_texts(spans, role):\\n    return [\\n        normalize_display_text(x[\\"text\\"])\\n        for x in spans.get(role, [])\\n    ]\\n", "models/slr09/code/active_nara_adapter/silar_ai_adapter.py": "\\"\\"\\"Adapter SLR09 untuk keluaran inferensi TanggapMakna/SILAR yang tersedia.\\n\\nHanya memakai standard library. Tidak memuat model, menjalankan training,\\nmenulis database, atau melakukan Human Verification. INFORMATION-ARG dipetakan\\nke impacts sebagai usulan AI sesuai pilihan A pengguna.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport math\\nimport re\\nfrom typing import Any\\n\\nfrom silar_postprocessing import (\\n    POSTPROCESSING_VERSION, filter_negated_fallback_times, filter_spatial_information,\\n)\\nfrom silar_nara_composition import compose_fields\\n\\n\\nADAPTER_VERSION = \\"silar-adapter-0.4.0\\"\\nFUSION_VERSION = \\"silar-fusion-a4.22-0.1.0\\"\\nEVENT_CATEGORIES = (\\"FLOOD\\", \\"EARTHQUAKE\\", \\"FIRE\\", \\"ACCIDENT\\", \\"NON_EVENT\\")\\nNARA_LABELS = (\\n    \\"O\\", \\"ACCIDENT-EVENT\\", \\"AFFECTEDOBJECTS-ARG\\", \\"DEATHVICTIM-ARG\\",\\n    \\"EARTHQUAKE-EVENT\\", \\"FALSE-EVENT\\", \\"FIRE-EVENT\\", \\"FLOOD-EVENT\\",\\n    \\"INFORMATION-ARG\\", \\"OFFICER-ARG\\", \\"PLACE-ARG\\", \\"REASON-ARG\\",\\n    \\"STREET-ARG\\", \\"TIME-ARG\\", \\"WOUNDVICTIM-ARG\\",\\n)\\nEXTRACTION_FIELDS = (\\n    \\"location_text\\", \\"time_text\\", \\"occurred_at\\", \\"involved_parties\\",\\n    \\"impacts\\", \\"victims\\", \\"affected_objects\\",\\n)\\nWARNING_CODES = frozenset({\\n    \\"INFORMATION_AS_IMPACT_SUGGESTION\\", \\"TIME_FROM_DETERMINISTIC_FALLBACK\\",\\n})\\nTOKEN_PATTERN = re.compile(r\\"\\\\w+(?:[-\']\\\\w+)*|[^\\\\w\\\\s]\\", flags=re.UNICODE)\\n\\n\\nclass AdapterValidationError(ValueError):\\n    \\"\\"\\"Keluaran tidak memenuhi kontrak; caller harus menangani kegagalan.\\"\\"\\"\\n\\n\\nclass CalibrationRequiredError(AdapterValidationError):\\n    \\"\\"\\"Aturan confidence memerlukan konfigurasi kalibrasi eksplisit.\\"\\"\\"\\n\\n\\ndef _require(condition: bool, message: str) -> None:\\n    if not condition:\\n        raise AdapterValidationError(message)\\n\\n\\ndef _text(value: Any, name: str) -> str:\\n    _require(isinstance(value, str) and bool(value.strip()), f\\"{name}: string tidak kosong diperlukan\\")\\n    return value\\n\\n\\ndef _probability(value: Any, name: str) -> float:\\n    _require(type(value) in (int, float), f\\"{name}: angka diperlukan\\")\\n    number = float(value)\\n    _require(math.isfinite(number) and 0 <= number <= 1, f\\"{name}: harus dalam rentang 0..1\\")\\n    return number\\n\\n\\ndef _json_copy(value: Any, name: str = \\"raw_output\\", _active: set[int] | None = None) -> Any:\\n    \\"\\"\\"Salin JSON tanpa mengubah tipe atau mengizinkan NaN/objek arbitrer.\\"\\"\\"\\n    if value is None or type(value) in (str, bool, int):\\n        return value\\n    if type(value) is float:\\n        _require(math.isfinite(value), f\\"{name}: NaN/Infinity tidak diperbolehkan\\")\\n        return value\\n    _require(type(value) in (dict, list), f\\"{name}: nilai harus kompatibel JSON\\")\\n    active = set() if _active is None else _active\\n    _require(id(value) not in active, f\\"{name}: referensi melingkar tidak diperbolehkan\\")\\n    active.add(id(value))\\n    try:\\n        if isinstance(value, list):\\n            return [_json_copy(item, name, active) for item in value]\\n        _require(all(isinstance(key, str) for key in value), f\\"{name}: key harus string\\")\\n        return {key: _json_copy(item, name, active) for key, item in value.items()}\\n    finally:\\n        active.remove(id(value))\\n\\n\\ndef _model_version(model_version: str) -> str:\\n    version = f\\"{_text(model_version, \'model_version\')}|{ADAPTER_VERSION}\\"\\n    _require(len(version) <= 100, \\"model_version gabungan melebihi varchar(100)\\")\\n    return version\\n\\n\\ndef _display_text(text: str) -> str:\\n    \\"\\"\\"Normalisasi tampilan yang sama dengan helper notebook Fusion v1.1.\\"\\"\\"\\n    text = re.sub(r\\"\\\\s+([,;:!?])\\", r\\"\\\\1\\", text)\\n    text = re.sub(r\\"(?<=\\\\d)\\\\s*\\\\.\\\\s*(?=\\\\d)\\", \\".\\", text)\\n    text = re.sub(r\\"\\\\s+\\\\.\\", \\".\\", text)\\n    return re.sub(r\\"\\\\s{2,}\\", \\" \\", text).strip()\\n\\n\\ndef _grounded(text: str, narrative: str) -> bool:\\n    # Helper lama menyisipkan spasi di sekitar punctuation ketika merangkai\\n    # token. Cocokkan batas kata atau urutan token, bukan substring di tengah\\n    # kata. Pemeriksaan ini tetap tidak membuktikan kebenaran semantik.\\n    fragment = _display_text(text).casefold()\\n    source = _display_text(narrative).casefold()\\n    left = r\\"(?<!\\\\w)\\" if fragment[0].isalnum() or fragment[0] == \\"_\\" else \\"\\"\\n    right = r\\"(?!\\\\w)\\" if fragment[-1].isalnum() or fragment[-1] == \\"_\\" else \\"\\"\\n    if re.search(left + re.escape(fragment) + right, source):\\n        return True\\n    needle = [match.group(0).casefold() for match in TOKEN_PATTERN.finditer(text)]\\n    haystack = [match.group(0).casefold() for match in TOKEN_PATTERN.finditer(narrative)]\\n    return bool(needle) and any(\\n        haystack[index:index + len(needle)] == needle\\n        for index in range(len(haystack) - len(needle) + 1)\\n    )\\n\\n\\ndef _unique(values: list[str]) -> list[str]:\\n    result: list[str] = []\\n    for value in values:\\n        clean = _display_text(value)\\n        if clean not in result:\\n            result.append(clean)\\n    return result\\n\\n\\ndef _joined(values: list[str]) -> str | None:\\n    clean = _unique(values)\\n    return \\"; \\".join(clean) if clean else None\\n\\n\\ndef _read_sena(raw: Any) -> tuple[str, float]:\\n    _require(isinstance(raw, dict), \\"event_classifier harus object\\")\\n    category = raw.get(\\"label\\")\\n    _require(category in EVENT_CATEGORIES, \\"Label SENA tidak dikenali\\")\\n    confidence = _probability(raw.get(\\"confidence\\"), \\"SENA confidence\\")\\n    probabilities = raw.get(\\"probabilities\\")\\n    _require(isinstance(probabilities, dict), \\"SENA probabilities harus object\\")\\n    _require(set(probabilities) == set(EVENT_CATEGORIES), \\"SENA probabilities harus mencakup tepat lima kelas\\")\\n    probs = {key: _probability(value, f\\"probabilities.{key}\\") for key, value in probabilities.items()}\\n    _require(math.isclose(sum(probs.values()), 1.0, abs_tol=1e-6), \\"Jumlah probabilitas SENA tidak mendekati 1\\")\\n    _require(math.isclose(probs[category], confidence, abs_tol=1e-6), \\"Confidence tidak cocok dengan probabilitas kategori\\")\\n    _require(confidence + 1e-6 >= max(probs.values()), \\"Kategori SENA bukan kelas dengan probabilitas tertinggi\\")\\n    return category, confidence\\n\\n\\ndef _validate_tokens(tokens: Any, narrative: str) -> tuple[int, int]:\\n    _require(isinstance(tokens, list) and bool(tokens), \\"srl_token_predictions harus list tidak kosong\\")\\n    source_tokens = list(TOKEN_PATTERN.finditer(narrative))\\n    _require(len(tokens) <= len(source_tokens), \\"Jumlah prediksi NARA melebihi token narasi\\")\\n    for index, item in enumerate(tokens):\\n        _require(isinstance(item, dict), f\\"Token {index} harus object\\")\\n        source = source_tokens[index]\\n        _require(type(item.get(\\"start\\")) is int and type(item.get(\\"end\\")) is int, f\\"Offset token {index} harus integer\\")\\n        _require((item[\\"start\\"], item[\\"end\\"]) == source.span(), f\\"Offset token {index} tidak cocok dengan narasi\\")\\n        _require(item.get(\\"original\\") == source.group(0), f\\"Original token {index} tidak cocok dengan narasi\\")\\n        _require(item.get(\\"token\\") == source.group(0).lower(), f\\"Token lowercase {index} tidak cocok\\")\\n        _require(item.get(\\"label\\") in NARA_LABELS, f\\"Label NARA pada token {index} tidak dikenali\\")\\n        _probability(item.get(\\"confidence\\"), f\\"NARA confidence token {index}\\")\\n    return len(tokens), len(source_tokens)\\n\\n\\ndef _read_spans(spans: Any, narrative: str) -> dict[str, list[str]]:\\n    _require(isinstance(spans, dict), \\"srl_spans harus object\\")\\n    expected = set(NARA_LABELS) - {\\"O\\"}\\n    _require(set(spans) == expected, \\"srl_spans harus mencakup tepat 14 role non-O\\")\\n    values: dict[str, list[str]] = {}\\n    for role in NARA_LABELS[1:]:\\n        entries = spans[role]\\n        _require(isinstance(entries, list), f\\"Span {role} harus list\\")\\n        texts: list[str] = []\\n        for entry in entries:\\n            _require(isinstance(entry, dict) and entry.get(\\"role\\") == role, f\\"Object span {role} tidak valid\\")\\n            text = _text(entry.get(\\"text\\"), f\\"Span {role}.text\\")\\n            _probability(entry.get(\\"mean_confidence\\"), f\\"Span {role}.mean_confidence\\")\\n            _require(_grounded(text, narrative), f\\"Span {role} tidak ditemukan pada narasi\\")\\n            texts.append(text)\\n        values[role] = _unique(texts)\\n    return values\\n\\n\\ndef _validate_extraction(data: Any) -> None:\\n    _require(isinstance(data, dict) and set(data) == set(EXTRACTION_FIELDS), \\"normalized_output harus tepat tujuh field A4.22\\")\\n    for field in (\\"location_text\\", \\"time_text\\"):\\n        _require(data[field] is None or (isinstance(data[field], str) and bool(data[field].strip())), f\\"{field}: string tidak kosong atau null diperlukan\\")\\n    # Versi ini hanya memproyeksikan teks; belum mempunyai normalizer timestamp.\\n    _require(data[\\"occurred_at\\"] is None, \\"Adapter ini belum mendukung konversi occurred_at\\")\\n    for field in (\\"involved_parties\\", \\"impacts\\", \\"victims\\", \\"affected_objects\\"):\\n        _require(isinstance(data[field], list), f\\"{field} harus array\\")\\n        _require(all(isinstance(item, str) and bool(item.strip()) for item in data[field]), f\\"Item {field} harus string tidak kosong\\")\\n\\n\\ndef adapt_legacy_ai_output(\\n    legacy_output: dict[str, Any],\\n    *,\\n    narrative_id: str,\\n    sena_model_version: str,\\n    nara_model_version: str,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Proyeksikan keluaran analyze_report lama tanpa menjalankan komponennya.\\n\\n    Input wajib: report_text, event_classifier, srl_token_predictions, srl_spans.\\n    srl_spans harus berasal dari postprocessing/guard caller yang versinya\\n    dicatat pada trace runtime. Fungsi ini memvalidasi bentuk dan grounding, bukan mengulang\\n    guard korban atau mendeteksi seluruh konflik semantik.\\n    \\"\\"\\"\\n    _text(narrative_id, \\"narrative_id\\")\\n    raw = _json_copy(legacy_output, \\"legacy_output\\")\\n    _require(isinstance(raw, dict), \\"legacy_output harus object\\")\\n    narrative = _text(raw.get(\\"report_text\\"), \\"report_text\\")\\n    category, confidence = _read_sena(raw.get(\\"event_classifier\\"))\\n    processed, expected = _validate_tokens(raw.get(\\"srl_token_predictions\\"), narrative)\\n    roles = _read_spans(raw.get(\\"srl_spans\\"), narrative)\\n    warnings: list[dict[str, str]] = []\\n    filtered_values: list[dict[str, str]] = []\\n    times = roles[\\"TIME-ARG\\"]\\n    time_source = \\"NARA\\" if times else \\"NOT_FOUND\\"\\n\\n    # Hanya menerima hasil fallback yang sudah dinyatakan oleh pipeline lama.\\n    # Tidak menjalankan regex waktu baru atau menebak timestamp di adapter ini.\\n    legacy_card = raw.get(\\"incident_card\\")\\n    if not times and isinstance(legacy_card, dict):\\n        fallback = legacy_card.get(\\"time\\", [])\\n        _require(isinstance(fallback, list), \\"incident_card.time harus list\\")\\n        if fallback:\\n            _require(legacy_card.get(\\"time_source\\") == \\"DETERMINISTIC_FALLBACK\\", \\"Waktu tanpa TIME-ARG harus mempunyai asal fallback yang jelas\\")\\n            for text in fallback:\\n                _text(text, \\"time fallback\\")\\n                _require(_grounded(text, narrative), \\"Time fallback tidak ditemukan pada narasi\\")\\n            times, removed = filter_negated_fallback_times(narrative, _unique(fallback))\\n            filtered_values.extend(removed)\\n            if times:\\n                time_source = \\"DETERMINISTIC_FALLBACK\\"\\n                warnings.append({\\"code\\": \\"TIME_FROM_DETERMINISTIC_FALLBACK\\", \\"field\\": \\"time_text\\"})\\n\\n    impacts, removed = filter_spatial_information(narrative, roles[\\"INFORMATION-ARG\\"])\\n    filtered_values.extend(removed)\\n    if impacts:\\n        warnings.append({\\"code\\": \\"INFORMATION_AS_IMPACT_SUGGESTION\\", \\"field\\": \\"impacts\\"})\\n    normalized = {\\n        \\"location_text\\": _joined(roles[\\"PLACE-ARG\\"] + roles[\\"STREET-ARG\\"]),\\n        \\"time_text\\": _joined(times),\\n        \\"occurred_at\\": None,\\n        \\"involved_parties\\": roles[\\"OFFICER-ARG\\"],\\n        \\"impacts\\": impacts,\\n        \\"victims\\": _unique(roles[\\"DEATHVICTIM-ARG\\"] + roles[\\"WOUNDVICTIM-ARG\\"]),\\n        \\"affected_objects\\": roles[\\"AFFECTEDOBJECTS-ARG\\"],\\n    }\\n    normalized, composition = compose_fields(narrative, raw[\\"srl_token_predictions\\"], normalized)\\n    _validate_extraction(normalized)\\n    return {\\n        \\"source_context\\": {\\n            \\"narrative_id\\": narrative_id,\\n            \\"narrative_sha256\\": hashlib.sha256(narrative.encode(\\"utf-8\\")).hexdigest(),\\n        },\\n        \\"sena_result\\": {\\n            \\"narrative_id\\": narrative_id,\\n            \\"category\\": category,\\n            \\"confidence\\": confidence,\\n            \\"raw_output\\": _json_copy(raw[\\"event_classifier\\"]),\\n            \\"model_version\\": _model_version(sena_model_version),\\n        },\\n        \\"nara_result\\": {\\n            \\"narrative_id\\": narrative_id,\\n            \\"raw_output\\": {\\n                \\"srl_token_predictions\\": _json_copy(raw[\\"srl_token_predictions\\"]),\\n                \\"srl_spans\\": _json_copy(raw[\\"srl_spans\\"]),\\n            },\\n            \\"normalized_output\\": _json_copy(normalized),\\n            \\"normalization_warnings\\": warnings,\\n            \\"model_version\\": _model_version(nara_model_version),\\n        },\\n        \\"adapter_metadata\\": {\\n            \\"adapter_version\\": ADAPTER_VERSION,\\n            \\"information_arg_policy\\": \\"A_IMPACT_SUGGESTION\\",\\n            \\"postprocessing_version\\": POSTPROCESSING_VERSION,\\n            \\"filtered_values\\": filtered_values,\\n            \\"nara_tokens_processed\\": processed,\\n            \\"nara_source_tokens\\": expected,\\n            \\"nara_token_coverage_complete\\": processed == expected,\\n            \\"time_source\\": time_source,\\n            \\"timestamp_normalization_supported\\": False,\\n            \\"extraction_ambiguity_detection_supported\\": False,\\n            \\"span_guard_responsibility\\": \\"CALLER_POSTPROCESSING\\",\\n            \\"composition\\": composition,\\n        },\\n    }\\n\\n\\ndef fuse_for_silar(\\n    adapted: dict[str, Any],\\n    *,\\n    confidence_threshold: float | None,\\n    calibration_reference: str | None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Susun field kartu; threshold wajib dipasok dari kalibrasi yang sah.\\n\\n    Referensi kalibrasi adalah metadata penelusuran. Keberadaan string tersebut\\n    tidak membuktikan bahwa pengukuran kalibrasi telah dilakukan.\\n    \\"\\"\\"\\n    if confidence_threshold is None or not isinstance(calibration_reference, str) or not calibration_reference.strip():\\n        raise CalibrationRequiredError(\\"Threshold dan referensi kalibrasi wajib ditetapkan sebelum Fusion diaktifkan\\")\\n    threshold = _probability(confidence_threshold, \\"confidence_threshold\\")\\n    bundle = _json_copy(adapted, \\"adapted\\")\\n    _require(isinstance(bundle, dict), \\"adapted harus object\\")\\n    source = bundle.get(\\"source_context\\")\\n    sena = bundle.get(\\"sena_result\\")\\n    nara = bundle.get(\\"nara_result\\")\\n    metadata = bundle.get(\\"adapter_metadata\\")\\n    _require(all(isinstance(item, dict) for item in (source, sena, nara, metadata)), \\"Envelope adapter tidak lengkap\\")\\n    narrative_id = _text(source.get(\\"narrative_id\\"), \\"source narrative_id\\")\\n    _require(sena.get(\\"narrative_id\\") == nara.get(\\"narrative_id\\") == narrative_id, \\"Sumber narasi SENA dan NARA berbeda\\")\\n    _require(bool(re.fullmatch(r\\"[0-9a-f]{64}\\", str(source.get(\\"narrative_sha256\\", \\"\\")))), \\"Hash narasi tidak valid\\")\\n    _require(metadata.get(\\"adapter_version\\") == ADAPTER_VERSION, \\"Versi adapter tidak didukung\\")\\n    category, confidence = _read_sena(sena.get(\\"raw_output\\"))\\n    _require(sena.get(\\"category\\") == category and sena.get(\\"confidence\\") == confidence, \\"Proyeksi SENA berbeda dari raw output\\")\\n    data = nara.get(\\"normalized_output\\")\\n    _validate_extraction(data)\\n    warnings = nara.get(\\"normalization_warnings\\")\\n    _require(isinstance(warnings, list), \\"normalization_warnings harus list\\")\\n    for warning in warnings:\\n        _require(isinstance(warning, dict) and set(warning) == {\\"code\\", \\"field\\"}, \\"Bentuk normalization warning tidak valid\\")\\n        _require(warning[\\"code\\"] in WARNING_CODES and warning[\\"field\\"] in EXTRACTION_FIELDS, \\"Kode warning/kemampuan ambiguity belum didukung adapter ini\\")\\n    reasons = [\\"LOW_CLASSIFICATION_CONFIDENCE\\"] if confidence < threshold else []\\n    card_data = {\\"event_type\\": None if category == \\"NON_EVENT\\" else category, **_json_copy(data)}\\n    return {\\n        \\"incident_card_version_fields\\": {\\n            \\"data\\": card_data,\\n            \\"uncertain\\": bool(reasons),\\n            \\"uncertain_reasons\\": reasons,\\n            \\"fusion_version\\": FUSION_VERSION,\\n        },\\n        \\"integration_signals\\": {\\n            \\"blocked_by_non_event\\": category == \\"NON_EVENT\\",\\n        },\\n        \\"fusion_metadata\\": {\\n            \\"confidence_threshold\\": threshold,\\n            \\"calibration_reference\\": calibration_reference,\\n            \\"extraction_ambiguity_detection_supported\\": False,\\n        },\\n    }\\n", "models/slr09/code/active_nara_adapter/silar_nara_composition.py": "\\"\\"\\"Komposisi terbatas dari role NARA dan teks sumber, tanpa inferensi baru.\\n\\nTidak menebak role O, kategori SENA, korban, atau timestamp. Hanya span yang\\nlolos proyeksi/guard baseline dan punya satu lokasi literal yang digunakan.\\n\\"\\"\\"\\nfrom __future__ import annotations\\nfrom copy import deepcopy\\nimport re\\n\\nCOMPOSITION_VERSION = \\"silar-nara-composition-0.1.0\\"\\nROLE_GUIDELINE_VERSION = \\"nara-role-guideline-0.2\\"\\nTOKEN = re.compile(r\\"\\\\w+(?:[-\']\\\\w+)*|[^\\\\w\\\\s]\\", re.UNICODE)\\nCOMPONENT_FIELDS = {\\n    \\"impacts\\": \\"INFORMATION-ARG\\", \\"affected_objects\\": \\"AFFECTEDOBJECTS-ARG\\",\\n    \\"involved_parties\\": \\"OFFICER-ARG\\", \\"victims\\": \\"GUARDED_VICTIM\\",\\n}\\nQUANTITIES = frozenset({\\"satu\\", \\"dua\\", \\"tiga\\", \\"empat\\", \\"lima\\", \\"enam\\", \\"tujuh\\", \\"delapan\\",\\n                        \\"sembilan\\", \\"sepuluh\\", \\"sebelas\\", \\"belasan\\", \\"puluhan\\", \\"ratusan\\",\\n                        \\"ribuan\\", \\"beberapa\\", \\"sejumlah\\"})\\nPERSON_NOUNS = frozenset({\\"orang\\", \\"warga\\", \\"korban\\", \\"petugas\\", \\"penumpang\\", \\"pekerja\\",\\n                          \\"anak\\", \\"siswa\\", \\"pelajar\\", \\"mahasiswa\\", \\"karyawan\\"})\\n\\n\\ndef locate(phrase, source, processed_tokens=None):\\n    \\"\\"\\"Offset char/token untuk satu kemunculan; ambigu tetap tanpa perluasan.\\"\\"\\"\\n    words = [m.group(0).casefold() for m in TOKEN.finditer(phrase)]\\n    available = list(TOKEN.finditer(source))\\n    if processed_tokens is not None:\\n        available = available[:processed_tokens]\\n    haystack = [m.group(0).casefold() for m in available]\\n    starts = [i for i in range(len(haystack)-len(words)+1) if words and haystack[i:i+len(words)] == words]\\n    if len(starts) != 1:\\n        return None\\n    first, last = starts[0], starts[0]+len(words)-1\\n    return {\\"start_token\\": first, \\"end_token\\": last, \\"start\\": available[first].start(),\\n            \\"end\\": available[last].end(), \\"text\\": source[available[first].start():available[last].end()]}\\n\\n\\ndef _trace(field, rule, span, components):\\n    return {\\"field\\": field, \\"rule\\": rule, \\"value\\": span[\\"text\\"], \\"start\\": span[\\"start\\"],\\n            \\"end\\": span[\\"end\\"], \\"components\\": components, \\"origin\\": \\"DETERMINISTIC_SOURCE_COMPOSITION\\"}\\n\\n\\ndef compose_fields(narrative, token_predictions, normalized):\\n    \\"\\"\\"Komposisi hanya memakai field baseline sebagai bukti role yang lolos guard.\\n\\n    Nilai input tidak diubah. Tidak mengisi lokasi dari token O maupun korban\\n    dari kata ambulans. Prefix negasi/waktu yang sudah dihapus tidak dipulihkan.\\n    \\"\\"\\"\\n    result = deepcopy(normalized)\\n    traces = []\\n    processed = len(token_predictions)\\n    source_tokens = list(TOKEN.finditer(narrative))\\n    blocks = []\\n    for field, role in COMPONENT_FIELDS.items():\\n        for value in normalized[field]:\\n            span = locate(value, narrative, processed)\\n            if span:\\n                blocks.append({**span, \\"field\\": field, \\"role\\": role})\\n    # Pakai frasa yang diproyeksikan/diamankan, bukan role korban mentah yang\\n    # bisa telah dibuang guard. Hanya rentang bersebelahan yang digabung.\\n    impacts = []\\n    for value in normalized[\\"impacts\\"]:\\n        anchor = next((b for b in blocks if b[\\"field\\"] == \\"impacts\\" and\\n                       b[\\"text\\"].casefold() == value.casefold()), None)\\n        if anchor is None:\\n            impacts.append(value)\\n            continue\\n        group = [anchor]\\n        left, right = anchor[\\"start_token\\"], anchor[\\"end_token\\"]\\n        while True:\\n            neighbors = [b for b in blocks if b not in group and\\n                         (b[\\"end_token\\"]+1 == left or b[\\"start_token\\"] == right+1)]\\n            if not neighbors:\\n                break\\n            # Satu label per token; guard bisa membentuk span lebih panjang.\\n            # Span bertabrakan tidak dipakai untuk menambah komposisi.\\n            added = False\\n            for block in neighbors:\\n                if any(not (block[\\"end_token\\"] < g[\\"start_token\\"] or block[\\"start_token\\"] > g[\\"end_token\\"]) for g in group):\\n                    continue\\n                # Jangan melintasi punctuation/konjungsi O. Token di setiap\\n                # komponen harus tersambung tanpa token di antaranya.\\n                group.append(block)\\n                left, right = min(left, block[\\"start_token\\"]), max(right, block[\\"end_token\\"])\\n                added = True\\n            if not added:\\n                break\\n        phrase = narrative[source_tokens[left].start():source_tokens[right].end()]\\n        if phrase not in impacts:\\n            impacts.append(phrase)\\n        if len(group) > 1:\\n            span = {\\"text\\": phrase, \\"start\\": source_tokens[left].start(), \\"end\\": source_tokens[right].end()}\\n            traces.append(_trace(\\"impacts\\", \\"ADJACENT_ROLE_SPANS\\", span,\\n                                 [{\\"role\\": b[\\"role\\"], \\"text\\": b[\\"text\\"], \\"start\\": b[\\"start\\"], \\"end\\": b[\\"end\\"]} for b in sorted(group, key=lambda b:b[\\"start\\"])]))\\n    result[\\"impacts\\"] = list(dict.fromkeys(impacts))\\n\\n    # Kata sejak/sekitar mengubah makna temporal; pertahankan jika tepat\\n    # bersebelahan dengan span waktu dan lokasi frasanya tidak ambigu.\\n    if normalized[\\"time_text\\"]:\\n        times = []\\n        for value in normalized[\\"time_text\\"].split(\\"; \\"):\\n            span = locate(value, narrative, processed)\\n            if span and span[\\"start_token\\"] > 0:\\n                previous = source_tokens[span[\\"start_token\\"]-1]\\n                if previous.group(0).casefold() in {\\"sejak\\", \\"sekitar\\"}:\\n                    extended = {\\"text\\": narrative[previous.start():span[\\"end\\"]], \\"start\\": previous.start(), \\"end\\": span[\\"end\\"]}\\n                    times.append(extended[\\"text\\"])\\n                    traces.append(_trace(\\"time_text\\", \\"LITERAL_TIME_MODIFIER\\", extended,\\n                                         [{\\"role\\": \\"TIME_SOURCE\\", \\"text\\": span[\\"text\\"], \\"start\\": span[\\"start\\"], \\"end\\": span[\\"end\\"]}]))\\n                    continue\\n            times.append(value)\\n        result[\\"time_text\\"] = \\"; \\".join(dict.fromkeys(times))\\n\\n    # Cakupan pihak berasal dari frasa korban yang sudah lolos pengamanan.\\n    # Ambil hanya prefix jumlah+nomina manusia, tanpa menebak identitas.\\n    for value in normalized[\\"victims\\"]:\\n        span = locate(value, narrative, processed)\\n        if not span:\\n            continue\\n        tokens = list(TOKEN.finditer(span[\\"text\\"]))\\n        words = [m.group(0).casefold() for m in tokens]\\n        length = 1 if words and words[0] == \\"seorang\\" else 0\\n        if len(words) >= 2 and (words[0].isdigit() or words[0] in QUANTITIES) and words[1] in PERSON_NOUNS:\\n            length = 2\\n        if not length:\\n            continue\\n        end = span[\\"start\\"] + tokens[length-1].end()\\n        party = narrative[span[\\"start\\"]:end]\\n        if party not in result[\\"involved_parties\\"]:\\n            result[\\"involved_parties\\"].append(party)\\n            traces.append(_trace(\\"involved_parties\\", \\"PERSON_PREFIX_FROM_GUARDED_VICTIM\\",\\n                                 {\\"text\\": party, \\"start\\": span[\\"start\\"], \\"end\\": end},\\n                                 [{\\"role\\": \\"GUARDED_VICTIM\\", \\"text\\": span[\\"text\\"], \\"start\\": span[\\"start\\"], \\"end\\": span[\\"end\\"]}]))\\n    return result, {\\"version\\": COMPOSITION_VERSION, \\"guideline_reference\\": ROLE_GUIDELINE_VERSION,\\n                    \\"rules_applied\\": traces, \\"new_neural_inference_executed\\": False,\\n                    \\"training_policy_of_input_model_verified\\": False}\\n", "models/slr09/code/active_nara_adapter/silar_postprocessing.py": "\\"\\"\\"Filter normalisasi SLR09 yang terbatas pada dua temuan review.\\n\\nTidak mengubah token/span mentah, menambah hasil ekstraksi, atau menentukan\\nkategori/uncertain. Aturan sengaja konservatif; bukan pendeteksi semantik umum.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\n\\nPOSTPROCESSING_VERSION = \\"silar-postprocessing-0.2.0\\"\\n_TOKEN = re.compile(r\\"\\\\w+(?:[-\']\\\\w+)*|[^\\\\w\\\\s]\\", flags=re.UNICODE)\\n_NEGATED_TIME_PREFIXES = (\\n    (\\"bukan\\",), (\\"bukan\\", \\"pada\\"),\\n    (\\"bukan\\", \\"kejadian\\"), (\\"bukan\\", \\"kejadian\\", \\"pada\\"),\\n    (\\"bukan\\", \\"peristiwa\\"), (\\"bukan\\", \\"peristiwa\\", \\"pada\\"),\\n    (\\"bukan\\", \\"terjadi\\"), (\\"bukan\\", \\"terjadi\\", \\"pada\\"),\\n    (\\"tidak\\", \\"terjadi\\"), (\\"tidak\\", \\"terjadi\\", \\"pada\\"),\\n    (\\"tak\\", \\"terjadi\\"), (\\"tak\\", \\"terjadi\\", \\"pada\\"),\\n)\\n_SPATIAL_CUES = frozenset({\\"di\\", \\"ke\\", \\"dari\\", \\"dekat\\", \\"depan\\", \\"belakang\\"})\\n# Hanya frasa lokasi sederhana yang dapat dikenali tanpa menebak kata baru.\\n# Kata predikat/kalimat tambahan menyebabkan span dipertahankan untuk review.\\n_LOCATION_HEADS = frozenset({\\"mall\\", \\"mal\\", \\"pasar\\", \\"pom\\", \\"bensin\\", \\"jalan\\"})\\n_LOCATION_WORDS = _LOCATION_HEADS | _SPATIAL_CUES | {\\"arah\\", \\"sekitar\\"}\\n\\n\\ndef _words(text: str) -> list[str]:\\n    return [match.group(0).casefold() for match in _TOKEN.finditer(text)]\\n\\n\\ndef _occurrences(source: list[str], value: list[str]) -> list[int]:\\n    if not value:\\n        return []\\n    return [index for index in range(len(source) - len(value) + 1)\\n            if source[index:index + len(value)] == value]\\n\\n\\ndef filter_negated_fallback_times(\\n    narrative: str, values: list[str],\\n) -> tuple[list[str], list[dict[str, str]]]:\\n    \\"\\"\\"Buang fallback hanya jika semua kemunculannya disangkal eksplisit.\\n\\n    Contoh: \'bukan kejadian hari ini\'. Negasi korban di bagian lain kalimat\\n    tidak berlaku sebagai negasi waktu. TIME-ARG model tidak diproses di sini.\\n    \\"\\"\\"\\n    source = _words(narrative)\\n    kept, removed = [], []\\n    for value in values:\\n        starts = _occurrences(source, _words(value))\\n        denied = [any(index >= len(prefix) and source[index-len(prefix):index] == list(prefix)\\n                      for prefix in _NEGATED_TIME_PREFIXES) for index in starts]\\n        if starts and all(denied):\\n            removed.append({\\"field\\": \\"time_text\\", \\"text\\": value,\\n                            \\"rule\\": \\"EXPLICIT_NEGATION_OF_FALLBACK_TIME\\"})\\n        else:\\n            kept.append(value)\\n    return kept, removed\\n\\n\\ndef filter_spatial_information(\\n    narrative: str, values: list[str],\\n) -> tuple[list[str], list[dict[str, str]]]:\\n    \\"\\"\\"Buang INFORMATION yang hanya lokasi sederhana dalam konteks spasial.\\n\\n    Contoh: \'mall\' pada \'dari jalan depan mall\', atau \'depan pom bensin\'.\\n    \'mall rusak\' tetap disimpan karena filter tidak menghapus frasa dampak.\\n    Kosakata di atas membatasi cakupan; lokasi lain belum tentu terdeteksi.\\n    \\"\\"\\"\\n    source = _words(narrative)\\n    kept, removed = [], []\\n    for value in values:\\n        words = _words(value)\\n        starts = _occurrences(source, words)\\n        simple_location = (bool(words) and len(words) <= 5\\n                           and bool(set(words) & _LOCATION_HEADS)\\n                           and all(word in _LOCATION_WORDS for word in words))\\n        internal_cue = bool(words and words[0] in _SPATIAL_CUES)\\n        spatial_occurrences = [internal_cue or (index > 0 and source[index-1] in _SPATIAL_CUES)\\n                               for index in starts]\\n        if simple_location and starts and all(spatial_occurrences):\\n            removed.append({\\"field\\": \\"impacts\\", \\"text\\": value,\\n                            \\"rule\\": \\"SPATIAL_INFORMATION_NOT_IMPACT\\"})\\n        else:\\n            kept.append(value)\\n    return kept, removed\\n", "models/slr09/code/active_nara_adapter/silar_victim_guard.py": "\\"\\"\\"Tambahan guard sesuai keputusan A: kondisi fisik eksplisit tetap literal.\\n\\nGuard historis tetap dipakai untuk death dan wound lama. Modul ini hanya\\nmenangani span WOUND yang memuat kritis/sesak napas/tidak sadarkan diri dan\\nbelum memuat cue luka lama. Tidak membuat span dari token O atau inferensi\\nkendaraan/ambulans/hilang/pengungsi. Prediksi mentah tidak diubah.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom copy import deepcopy\\nimport re\\n\\nimport legacy_postprocessing as legacy\\n\\nVICTIM_GUARD_VERSION = \\"silar-victim-guard-0.4.0\\"\\nROLE_GUIDELINE_VERSION = \\"nara-role-guideline-0.2\\"\\n\\nCONDITIONS = re.compile(\\n    r\\"\\\\b(?:kritis|sesak\\\\s+(?:napas|nafas)|(?:tak|tidak)\\\\s+(?:sadar|sadarkan)\\\\s+diri)\\\\b\\", re.I)\\nPERSON = re.compile(\\n    r\\"\\\\b(?:korban|orang|warga|petugas|penumpang|pengendara|angkotpengendara|\\"\\n    r\\"sopir|supir|pelajar|siswa|siswi|anak|balita|pekerja|pria|wanita)\\\\b\\", re.I)\\nQUANTITY = re.compile(r\\"\\\\b(?:\\\\d+|satu|dua|tiga|empat|lima|enam|tujuh|delapan|sembilan|sepuluh)\\\\s*$\\", re.I)\\nINCIDENT_CONTEXT = re.compile(r\\"\\\\b(?:kecelakaan|kebakaran|banjir|gempa|tabrak\\\\w*|tewas|korban)\\\\b\\", re.I)\\nNEGATED_PREFIX = re.compile(\\n    r\\"\\\\b(?:tidak|tak|bukan|tanpa|nihil|belum)\\\\s+(?:(?:ada|terdapat|seorang|satupun|satu|korban|\\"\\n    r\\"orang|warga|yang|mengalami|dalam|keadaan|berada|dinyatakan|dilaporkan|masih|lagi|sedang|pernah|sampai)\\\\s*)*$\\", re.I)\\n\\n\\ndef _fold(text):\\n    return \\" \\".join(str(text).casefold().split())\\n\\n\\ndef _allowed_condition(text, span):\\n    phrase = span.get(\\"text\\", \\"\\")\\n    folded_phrase, folded_text = _fold(phrase), _fold(text)\\n    if not folded_phrase:\\n        return False\\n    starts = [m.start() for m in re.finditer(re.escape(folded_phrase), folded_text)]\\n    if not starts:\\n        return False\\n    conditions = list(CONDITIONS.finditer(folded_phrase))\\n    for condition in conditions:\\n        # Kata \\"tak/tidak\\" yang berada DI DALAM cue tidak sadar adalah kondisi,\\n        # bukan negasi kondisi. Negasi luarnya diperiksa pada prefix sebelum cue.\\n        local_prefix = folded_phrase[:condition.start()]\\n        person = bool(PERSON.search(folded_phrase))\\n        quantified_incident = bool(QUANTITY.search(local_prefix) and INCIDENT_CONTEXT.search(text))\\n        if not (person or quantified_incident):\\n            continue\\n        occurrence_denials = []\\n        for start in starts:\\n            prior = folded_text[:start + condition.start()]\\n            prior = re.split(r\\"[.;!?]\\", prior)[-1]\\n            # Klausa tandingan memulai konteks baru; negasi klausa lain tidak\\n            # digunakan untuk menolak korban yang dinyatakan positif.\\n            prior = re.split(r\\"\\\\b(?:tetapi|namun|sedangkan)\\\\b\\", prior)[-1]\\n            prior = \\" \\".join(prior.split()[-9:])\\n            occurrence_denials.append(bool(NEGATED_PREFIX.search(prior + \\" \\")))\\n        if not any(occurrence_denials):\\n            return True\\n    return False\\n\\n\\ndef apply_reviewed_victim_safety(text, spans):\\n    \\"\\"\\"Perluasan terbatas untuk kondisi yang disetujui, tanpa relabeling.\\"\\"\\"\\n    baseline_input = deepcopy(spans)\\n    special, ordinary = [], []\\n    for span in baseline_input.get(\\"WOUNDVICTIM-ARG\\", []):\\n        if CONDITIONS.search(span.get(\\"text\\", \\"\\")) and not legacy.has_any_pattern(\\n                span.get(\\"text\\", \\"\\"), legacy.WOUND_EVIDENCE_PATTERNS):\\n            special.append(span)\\n        else:\\n            ordinary.append(span)\\n    baseline_input[\\"WOUNDVICTIM-ARG\\"] = ordinary\\n    guarded, flags = legacy.apply_victim_safety(text, baseline_input)\\n    kept = guarded.setdefault(\\"WOUNDVICTIM-ARG\\", [])\\n    known = {_fold(r.get(\\"text\\", \\"\\")) for r in kept}\\n    rejected = False\\n    for span in special:\\n        if _allowed_condition(text, span):\\n            key = _fold(span[\\"text\\"])\\n            if key not in known:\\n                kept.append(deepcopy(span))\\n                known.add(key)\\n        else:\\n            rejected = True\\n    if rejected and not kept and \\"WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE\\" not in flags:\\n        flags.append(\\"WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE\\")\\n    if kept:\\n        flags = [f for f in flags if f != \\"WOUND_VICTIM_SUPPRESSED_NO_WOUND_EVIDENCE\\"]\\n    return guarded, flags\\n", "models/slr09/code/active_nara_adapter/slr09_candidate_runtime.py": "\\"\\"\\"Pemuatan kandidat SENA/NARA terpilih; inferensi lokal tanpa training/backend.\\n\\nIdentitas berasal dari manifest ekspor kandidat, bukan asset_lock baseline.\\nModel hanya dimuat dari berkas lokal yang cocok hash; training_args.bin diabaikan.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport importlib.metadata\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport statistics\\nimport sys\\nimport time\\nimport zipfile\\n\\nfrom silar_ai_adapter import (\\n    ADAPTER_VERSION, EVENT_CATEGORIES, FUSION_VERSION, NARA_LABELS,\\n    adapt_legacy_ai_output, fuse_for_silar,\\n)\\nfrom slr09_runtime import (\\n    SilarModelRunner, build_adapter_input, environment_snapshot, read_records,\\n    sha256_file, text_digest, write_json,\\n)\\n\\nCANDIDATE_RUNNER_VERSION = \\"silar-candidate-runtime-0.2.0\\"\\nPACKAGE_ROOT = Path(__file__).resolve().parents[2]\\nRUNTIME_FILES = frozenset({\\n    \\"config.json\\", \\"model.safetensors\\", \\"tokenizer.json\\", \\"tokenizer_config.json\\",\\n    \\"special_tokens_map.json\\", \\"vocab.txt\\",\\n})\\n\\n\\ndef selected_assets(root=PACKAGE_ROOT):\\n    root = Path(root)\\n    config = json.loads((root / \\"config/SLR09_Konfigurasi_Integrasi_v0.1.json\\").read_text())\\n    lock = json.loads((root / \\"config/candidate_asset_lock.json\\").read_text())\\n    profile = json.loads((root / \\"config/inference_environment.json\\").read_text())\\n    if config.get(\\"adapter_version\\") != ADAPTER_VERSION or config.get(\\"fusion_version\\") != FUSION_VERSION:\\n        raise ValueError(\\"Versi adapter/Fusion tidak cocok dengan konfigurasi kandidat\\")\\n    threshold = config.get(\\"confidence_threshold\\")\\n    if type(threshold) not in (float, int) or not math.isfinite(threshold) or not 0 <= threshold <= 1:\\n        raise ValueError(\\"Threshold kandidat tidak valid\\")\\n    if not isinstance(config.get(\\"calibration_reference\\"), str) or not config[\\"calibration_reference\\"].strip():\\n        raise ValueError(\\"Rujukan pengukuran cutoff kandidat wajib tersedia\\")\\n    for name, expected_labels in ((\\"sena\\", EVENT_CATEGORIES), (\\"nara\\", NARA_LABELS)):\\n        spec, candidate = lock[\\"models\\"][name], config[name]\\n        for key in (\\"archive_sha256\\", \\"archive_name\\", \\"run_id\\", \\"labels\\"):\\n            if spec.get(key) != candidate.get(key):\\n                raise ValueError(f\\"{name}: lock kandidat dan konfigurasi berbeda pada {key}\\")\\n        if tuple(spec[\\"labels\\"]) != tuple(expected_labels):\\n            raise ValueError(f\\"{name}: urutan label kandidat tidak sesuai adapter\\")\\n        if spec[\\"files\\"].get(\\"model.safetensors\\") != candidate[\\"weights_sha256\\"]:\\n            raise ValueError(f\\"{name}: hash bobot kandidat tidak konsisten\\")\\n        if set(spec[\\"files\\"]) != RUNTIME_FILES:\\n            raise ValueError(f\\"{name}: daftar berkas runtime kandidat tidak dikenal\\")\\n        for value in (spec[\\"archive_sha256\\"], *spec[\\"files\\"].values()):\\n            if not isinstance(value, str) or len(value) != 64 or any(c not in \\"0123456789abcdef\\" for c in value):\\n                raise ValueError(f\\"{name}: hash SHA-256 tidak valid\\")\\n    return config, lock, profile\\n\\n\\ndef verify_candidate_directory(directory, name, spec):\\n    directory = Path(directory)\\n    if directory.is_symlink() or not directory.is_dir():\\n        raise ValueError(f\\"{name}: direktori kandidat lokal tidak tersedia\\")\\n    observed = {}\\n    for filename, expected in spec[\\"files\\"].items():\\n        if filename not in RUNTIME_FILES:\\n            raise ValueError(f\\"{name}: berkas di luar whitelist runtime\\")\\n        path = directory / filename\\n        if path.is_symlink() or not path.is_file():\\n            raise ValueError(f\\"{name}: berkas tidak tersedia: {filename}\\")\\n        actual = sha256_file(path)\\n        if actual != expected:\\n            raise ValueError(f\\"{name}: hash kandidat berbeda: {filename}\\")\\n        observed[filename] = actual\\n    model_config = json.loads((directory / \\"config.json\\").read_text())\\n    expected = {str(i): label for i, label in enumerate(spec[\\"labels\\"])}\\n    if model_config.get(\\"id2label\\") != expected:\\n        raise ValueError(f\\"{name}: label konfigurasi bobot tidak sesuai kandidat\\")\\n    return {\\"model_id\\": spec[\\"model_id\\"], \\"run_id\\": spec[\\"run_id\\"],\\n            \\"archive_sha256_expected\\": spec[\\"archive_sha256\\"], \\"files\\": observed,\\n            \\"labels\\": spec[\\"labels\\"], \\"candidate_directory_verified\\": True}\\n\\n\\ndef extract_candidate_archive(archive, destination, name, spec):\\n    archive, destination = Path(archive), Path(destination)\\n    if not archive.is_file() or sha256_file(archive) != spec[\\"archive_sha256\\"]:\\n        raise ValueError(f\\"{name}: arsip berbeda dari kandidat terpilih\\")\\n    if destination.is_symlink():\\n        raise ValueError(\\"Direktori ekstraksi tidak boleh symlink\\")\\n    destination.mkdir(parents=True, exist_ok=True)\\n    with zipfile.ZipFile(archive) as source:\\n        for filename, expected in spec[\\"files\\"].items():\\n            if filename not in RUNTIME_FILES or source.namelist().count(filename) != 1:\\n                raise ValueError(f\\"{name}: anggota runtime tidak tersedia/unik: {filename}\\")\\n            target = destination / filename\\n            if target.is_symlink():\\n                raise ValueError(\\"Target ekstraksi tidak boleh symlink\\")\\n            if target.exists():\\n                if not target.is_file() or sha256_file(target) != expected:\\n                    raise ValueError(f\\"{name}: target sudah ada dan berbeda: {filename}\\")\\n                continue\\n            with source.open(filename) as stream, target.open(\\"xb\\") as output:\\n                while chunk := stream.read(1024 * 1024):\\n                    output.write(chunk)\\n    evidence = verify_candidate_directory(destination, name, spec)\\n    evidence[\\"archive_sha256_observed\\"] = sha256_file(archive)\\n    return evidence\\n\\n\\ndef check_dependencies(profile, version_lookup=importlib.metadata.version):\\n    observed, problems = {}, []\\n    for name in profile[\\"guard_packages\\"]:\\n        try:\\n            observed[name] = version_lookup(name)\\n        except importlib.metadata.PackageNotFoundError:\\n            observed[name] = None\\n        expected = profile[\\"pinned_packages\\"][name]\\n        # Build CPU/CUDA PyTorch boleh berbeda suffix; versi pokok tetap sama.\\n        actual = observed[name]\\n        normalized = actual.split(\\"+\\", 1)[0] if actual else None\\n        if normalized != expected:\\n            problems.append(f\\"{name}: perlu {expected}, tersedia {actual or \'belum terpasang\'}\\")\\n    if problems:\\n        raise RuntimeError(\\"Profil inference belum sesuai: \\" + \\"; \\".join(problems))\\n    return observed\\n\\n\\nclass CandidateModelRunner(SilarModelRunner):\\n    \\"\\"\\"Memakai algoritma inferensi historis dengan identitas kandidat eksplisit.\\"\\"\\"\\n    def __init__(self, sena_dir, nara_dir, *, root=PACKAGE_ROOT, device=\\"cpu\\", cpu_threads=None):\\n        self.config, lock, profile = selected_assets(root)\\n        self.model_evidence = {name: verify_candidate_directory(directory, name, lock[\\"models\\"][name])\\n                               for name, directory in ((\\"sena\\", sena_dir), (\\"nara\\", nara_dir))}\\n        self.inference_versions = check_dependencies(profile)\\n        import torch\\n        from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification\\n        self.torch, self.device = torch, torch.device(device)\\n        if self.device.type not in {\\"cpu\\", \\"cuda\\"}:\\n            raise ValueError(\\"Device harus cpu atau cuda\\")\\n        if self.device.type == \\"cuda\\" and not torch.cuda.is_available():\\n            raise RuntimeError(\\"CUDA diminta tetapi tidak tersedia\\")\\n        if cpu_threads is not None:\\n            if type(cpu_threads) is not int or cpu_threads < 1:\\n                raise ValueError(\\"cpu_threads harus integer positif\\")\\n            torch.set_num_threads(cpu_threads)\\n        started = time.perf_counter()\\n        self.sena_tokenizer, self.sena_model = self._load(sena_dir, AutoTokenizer, AutoModelForSequenceClassification)\\n        self.nara_tokenizer, self.nara_model = self._load(nara_dir, AutoTokenizer, AutoModelForTokenClassification)\\n        self.sync()\\n        self.load_seconds = time.perf_counter() - started\\n\\n\\ndef candidate_record(record, event_prediction, nara_tokens, config, lock):\\n    raw, trace = build_adapter_input(record[\\"text\\"], event_prediction, nara_tokens)\\n    adapted = adapt_legacy_ai_output(raw, narrative_id=record[\\"id\\"],\\n        sena_model_version=lock[\\"models\\"][\\"sena\\"][\\"model_id\\"],\\n        nara_model_version=lock[\\"models\\"][\\"nara\\"][\\"model_id\\"])\\n    fused = fuse_for_silar(adapted, confidence_threshold=config[\\"confidence_threshold\\"],\\n        calibration_reference=config[\\"calibration_reference\\"])\\n    return {\\"id\\": record[\\"id\\"], \\"input_sha256\\": text_digest(record[\\"text\\"]),\\n            \\"event_classifier\\": event_prediction, \\"adapted_result\\": adapted,\\n            \\"fusion_result\\": fused, \\"postprocessing_trace\\": trace}\\n\\n\\ndef run_smoke(args):\\n    records = read_records(args.input)\\n    output = args.output_dir\\n    if output.exists() and any(output.iterdir()):\\n        raise ValueError(\\"Gunakan direktori hasil kosong untuk menjaga bukti sebelumnya\\")\\n    output.mkdir(parents=True, exist_ok=True)\\n    summary_path, predictions_path = output / \\"run_summary.json\\", output / \\"predictions.jsonl\\"\\n    summary = {\\"schema\\": \\"slr09-candidate-smoke-0.2\\", \\"runner_version\\": CANDIDATE_RUNNER_VERSION,\\n        \\"status\\": \\"RUNNING\\", \\"input_sha256\\": sha256_file(args.input),\\n        \\"environment_label\\": args.environment_label, \\"environment_scope\\": args.environment_scope,\\n        \\"environment\\": environment_snapshot(), \\"training_executed\\": False,\\n        \\"model_inference_executed\\": False, \\"sena_inference_executed\\": False,\\n        \\"nara_inference_executed\\": False, \\"fusion_executed\\": False,\\n        \\"records_expected\\": len(records), \\"records_completed\\": 0,\\n        \\"production_deployed\\": False, \\"quality_acceptance_approved_by_this_run\\": False,\\n        \\"scope\\": \\"ASSET_LOADING_AND_SINGLE_PASS_INFERENCE_NOT_BACKEND_OR_QUALITY_ACCEPTANCE\\"}\\n    write_json(summary_path, summary)\\n    started, latencies, runner = time.perf_counter(), [], None\\n    try:\\n        config, lock, _ = selected_assets(args.root)\\n        summary[\\"confidence_threshold\\"] = config[\\"confidence_threshold\\"]\\n        summary[\\"calibration_reference\\"] = config[\\"calibration_reference\\"]\\n        summary[\\"configuration_sha256\\"] = sha256_file(args.root / \\"config/SLR09_Konfigurasi_Integrasi_v0.1.json\\")\\n        summary[\\"candidate_lock_sha256\\"] = sha256_file(args.root / \\"config/candidate_asset_lock.json\\")\\n        runner = CandidateModelRunner(args.sena_dir, args.nara_dir, root=args.root,\\n                                      device=args.device, cpu_threads=args.cpu_threads)\\n        summary[\\"models\\"], summary[\\"model_load_seconds\\"] = runner.model_evidence, runner.load_seconds\\n        with predictions_path.open(\\"x\\", encoding=\\"utf-8\\") as stream:\\n            for record in records:\\n                runner.sync()\\n                before = time.perf_counter()\\n                event, sena_coverage = runner.predict_sena(record[\\"text\\"])\\n                summary[\\"model_inference_executed\\"] = summary[\\"sena_inference_executed\\"] = True\\n                tokens, nara_coverage = runner.predict_nara(record[\\"text\\"])\\n                summary[\\"nara_inference_executed\\"] = True\\n                result = candidate_record(record, event, tokens, config, lock)\\n                summary[\\"fusion_executed\\"] = True\\n                runner.sync()\\n                elapsed = time.perf_counter() - before\\n                result.update(sena_token_coverage=sena_coverage, nara_token_coverage=nara_coverage,\\n                              elapsed_seconds=elapsed)\\n                stream.write(json.dumps(result, ensure_ascii=False, allow_nan=False) + \\"\\\\n\\")\\n                stream.flush()\\n                latencies.append(elapsed)\\n                summary[\\"records_completed\\"] += 1\\n        summary[\\"status\\"] = \\"SUCCEEDED\\"\\n    except Exception as error:\\n        summary[\\"status\\"] = \\"FAILED\\"\\n        summary[\\"error\\"] = {\\"type\\": type(error).__name__, \\"message\\": str(error)}\\n        raise\\n    finally:\\n        summary[\\"wall_seconds_including_loading\\"] = time.perf_counter() - started\\n        if runner is not None:\\n            summary[\\"environment\\"] = environment_snapshot(runner.torch, runner.device)\\n        if latencies:\\n            summary[\\"per_record_seconds\\"] = {\\"min\\": min(latencies), \\"median\\": statistics.median(latencies),\\n                \\"max\\": max(latencies), \\"scope\\": \\"SINGLE_PASS_INCLUDING_FIRST_RECORD_NOT_CAPACITY_BENCHMARK\\"}\\n        if predictions_path.exists():\\n            summary[\\"predictions_sha256\\"] = sha256_file(predictions_path)\\n        write_json(summary_path, summary)\\n    return summary\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    extract = commands.add_parser(\\"extract\\", help=\\"Verifikasi dan ekstrak kedua arsip kandidat\\")\\n    extract.add_argument(\\"--root\\", type=Path, default=PACKAGE_ROOT)\\n    extract.add_argument(\\"--sena-zip\\", type=Path, required=True)\\n    extract.add_argument(\\"--nara-zip\\", type=Path, required=True)\\n    extract.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    run = commands.add_parser(\\"smoke\\", help=\\"Inferensi kedua kandidat dan adapter/Fusion terpilih\\")\\n    run.add_argument(\\"--root\\", type=Path, default=PACKAGE_ROOT)\\n    run.add_argument(\\"--sena-dir\\", type=Path, required=True)\\n    run.add_argument(\\"--nara-dir\\", type=Path, required=True)\\n    run.add_argument(\\"--input\\", type=Path, required=True)\\n    run.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    run.add_argument(\\"--device\\", choices=(\\"cpu\\", \\"cuda\\"), default=\\"cpu\\")\\n    run.add_argument(\\"--cpu-threads\\", type=int)\\n    run.add_argument(\\"--environment-label\\", required=True)\\n    run.add_argument(\\"--environment-scope\\", choices=(\\"TARGET\\", \\"REPRESENTATIVE\\"), required=True)\\n    args = parser.parse_args(argv)\\n    try:\\n        if args.command == \\"extract\\":\\n            _, lock, _ = selected_assets(args.root)\\n            if (args.output_dir / \\"extraction_evidence.json\\").exists():\\n                raise ValueError(\\"Bukti ekstraksi sudah ada; gunakan direktori hasil baru\\")\\n            evidence = {name: extract_candidate_archive(path, args.output_dir / name, name, lock[\\"models\\"][name])\\n                        for name, path in ((\\"sena\\", args.sena_zip), (\\"nara\\", args.nara_zip))}\\n            write_json(args.output_dir / \\"extraction_evidence.json\\", evidence)\\n        else:\\n            result = run_smoke(args)\\n            print(json.dumps({\\"status\\": result[\\"status\\"], \\"records_completed\\": result[\\"records_completed\\"]}, indent=2))\\n    except (ValueError, RuntimeError, OSError, KeyError) as error:\\n        print(f\\"Kandidat belum siap: {error}\\", file=sys.stderr)\\n        return 1\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    raise SystemExit(main())\\n", "models/slr09/code/active_nara_adapter/slr09_runtime.py": "\\"\\"\\"Runner baseline historis SLR09 dan helper inferensi.\\n\\nUntuk kandidat fine-tune baru gunakan slr09_candidate_runtime.py; identitas\\nasset_lock pada modul ini sengaja tetap milik baseline historis.\\n\\nInferensi saja; tidak melakukan training, menyimpan database, atau menjalankan\\nFusion/Completeness lama. Impor PyTorch/Transformers ditunda hingga model dimuat.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom copy import deepcopy\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport platform\\nimport shutil\\nimport statistics\\nimport sys\\nimport time\\nimport zipfile\\n\\nfrom silar_ai_adapter import (\\n    ADAPTER_VERSION, EVENT_CATEGORIES, NARA_LABELS, TOKEN_PATTERN,\\n    adapt_legacy_ai_output,\\n)\\n\\nRUNNER_VERSION = \\"silar-runtime-0.4.0\\"\\nMAX_LENGTH = 256\\nHERE = Path(__file__).resolve().parent\\n\\n\\ndef sha256_file(path):\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as stream:\\n        for block in iter(lambda: stream.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef text_digest(text):\\n    return hashlib.sha256(text.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n\\n\\ndef read_records(path):\\n    records, ids = [], set()\\n    with Path(path).open(encoding=\\"utf-8\\") as stream:\\n        for number, line in enumerate(stream, 1):\\n            if not line.strip():\\n                continue\\n            record = json.loads(line)\\n            if not isinstance(record, dict):\\n                raise ValueError(f\\"Baris {number}: object diperlukan\\")\\n            key, text = record.get(\\"id\\"), record.get(\\"text\\")\\n            if not isinstance(key, str) or not key.strip() or key in ids:\\n                raise ValueError(f\\"Baris {number}: ID kosong/duplikat/tidak valid\\")\\n            if not isinstance(text, str) or not TOKEN_PATTERN.search(text):\\n                raise ValueError(f\\"Baris {number}: narasi tanpa token\\")\\n            if \\"gold_label\\" in record and record[\\"gold_label\\"] not in EVENT_CATEGORIES:\\n                raise ValueError(f\\"Baris {number}: gold_label tidak dikenal\\")\\n            ids.add(key)\\n            records.append(record)\\n    if not records:\\n        raise ValueError(\\"File input kosong\\")\\n    return records\\n\\n\\ndef validate_prediction(prediction):\\n    if not isinstance(prediction, dict) or prediction.get(\\"label\\") not in EVENT_CATEGORIES:\\n        raise ValueError(\\"Prediksi SENA tidak valid\\")\\n    probabilities = prediction.get(\\"probabilities\\")\\n    if not isinstance(probabilities, dict) or set(probabilities) != set(EVENT_CATEGORIES):\\n        raise ValueError(\\"Probabilitas harus mencakup tepat lima kelas\\")\\n    numbers = [prediction.get(\\"confidence\\"), *probabilities.values()]\\n    if any(type(x) not in (int, float) or not math.isfinite(x) or not 0 <= x <= 1 for x in numbers):\\n        raise ValueError(\\"Confidence/probabilitas harus angka finite 0..1\\")\\n    confidence = prediction[\\"confidence\\"]\\n    if not math.isclose(sum(probabilities.values()), 1.0, abs_tol=1e-6):\\n        raise ValueError(\\"Jumlah probabilitas tidak mendekati satu\\")\\n    if not math.isclose(confidence, probabilities[prediction[\\"label\\"]], abs_tol=1e-6):\\n        raise ValueError(\\"Confidence berbeda dari probabilitas label\\")\\n    if confidence + 1e-6 < max(probabilities.values()):\\n        raise ValueError(\\"Label bukan argmax probabilitas\\")\\n\\n\\ndef asset_lock():\\n    return json.loads((HERE / \\"asset_lock.json\\").read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef verify_model_directory(directory, name):\\n    directory = Path(directory).resolve(strict=True)\\n    spec = asset_lock()[\\"models\\"][name]\\n    observed = {}\\n    for filename, expected_hash in spec[\\"files\\"].items():\\n        path = directory / filename\\n        if not path.is_file() or path.is_symlink():\\n            raise ValueError(f\\"{name}: berkas lokal tidak tersedia: {filename}\\")\\n        actual = sha256_file(path)\\n        if actual != expected_hash:\\n            raise ValueError(f\\"{name}: hash {filename} berbeda dari aset yang diaudit\\")\\n        observed[filename] = actual\\n    config = json.loads((directory / \\"config.json\\").read_text(encoding=\\"utf-8\\"))\\n    labels = tuple(config[\\"id2label\\"][str(i)] for i in range(len(spec[\\"labels\\"])))\\n    if labels != tuple(spec[\\"labels\\"]):\\n        raise ValueError(f\\"{name}: urutan label berbeda\\")\\n    return {\\"files\\": observed, \\"labels\\": list(labels), \\"model_id\\": spec[\\"model_id\\"]}\\n\\n\\ndef extract_model_archive(archive_path, destination, name):\\n    \\"\\"\\"Ekstrak hanya berkas runtime flat yang hash-nya telah dikunci.\\"\\"\\"\\n    archive_path, destination = Path(archive_path), Path(destination)\\n    spec = asset_lock()[\\"models\\"][name]\\n    if sha256_file(archive_path) != spec[\\"archive_sha256\\"]:\\n        raise ValueError(f\\"{name}: ZIP berbeda dari paket model yang diaudit\\")\\n    destination.mkdir(parents=True, exist_ok=True)\\n    with zipfile.ZipFile(archive_path) as archive:\\n        names = archive.namelist()\\n        for filename in spec[\\"files\\"]:\\n            if names.count(filename) != 1:\\n                raise ValueError(f\\"{name}: anggota ZIP tidak unik: {filename}\\")\\n            target = destination / filename\\n            if target.is_symlink():\\n                raise ValueError(\\"Target ekstraksi tidak boleh symlink\\")\\n            if target.exists():\\n                if sha256_file(target) != spec[\\"files\\"][filename]:\\n                    raise ValueError(f\\"Target sudah ada dan berbeda: {target}\\")\\n                continue\\n            with archive.open(filename) as source, target.open(\\"xb\\") as stream:\\n                shutil.copyfileobj(source, stream)\\n    # training_args.bin tidak dibaca atau dideserialisasi.\\n    return verify_model_directory(destination, name)\\n\\n\\ndef encode_nara_words(tokens, tokenizer, max_length=MAX_LENGTH):\\n    \\"\\"\\"Algoritma srl_encode_words notebook lama, dengan dependency eksplisit.\\"\\"\\"\\n    input_ids = [tokenizer.cls_token_id]\\n    positions = []\\n    partial_last_word = False\\n    for word in tokens:\\n        pieces = tokenizer.tokenize(str(word))\\n        if not pieces:\\n            pieces = [tokenizer.unk_token]\\n        piece_ids = tokenizer.convert_tokens_to_ids(pieces)\\n        remaining = max_length - 1 - len(input_ids)\\n        if remaining <= 0:\\n            break\\n        partial_last_word = len(piece_ids) > remaining\\n        piece_ids = piece_ids[:remaining]\\n        positions.append(len(input_ids))\\n        input_ids.extend(piece_ids)\\n    input_ids.append(tokenizer.sep_token_id)\\n    return {\\n        \\"input_ids\\": input_ids,\\n        \\"attention_mask\\": [1] * len(input_ids),\\n        \\"positions\\": positions,\\n        \\"partial_last_word\\": partial_last_word,\\n    }\\n\\n\\ndef build_adapter_input(text, event_prediction, nara_tokens):\\n    \\"\\"\\"Gunakan span/fallback historis dan guard v0.4 tanpa keputusan Fusion lama.\\"\\"\\"\\n    import legacy_postprocessing as legacy\\n    from silar_victim_guard import apply_reviewed_victim_safety, VICTIM_GUARD_VERSION\\n    spans_before_guard = legacy.all_srl_spans(nara_tokens)\\n    spans, safety_flags = apply_reviewed_victim_safety(text, deepcopy(spans_before_guard))\\n    time_values = legacy.span_texts(spans, \\"TIME-ARG\\")\\n    time_source = \\"SRL\\"\\n    if not time_values:\\n        time_values = [legacy.normalize_display_text(item[\\"text\\"]) for item in legacy.time_fallback(text)]\\n        time_source = \\"DETERMINISTIC_FALLBACK\\" if time_values else \\"MISSING\\"\\n    return {\\n        \\"report_text\\": text,\\n        \\"event_classifier\\": event_prediction,\\n        \\"srl_token_predictions\\": nara_tokens,\\n        \\"srl_spans\\": spans,\\n        \\"incident_card\\": {\\"time\\": time_values, \\"time_source\\": time_source},\\n    }, {\\"spans_before_guard\\": spans_before_guard, \\"victim_safety_flags\\": safety_flags,\\n        \\"victim_guard_version\\": VICTIM_GUARD_VERSION}\\n\\n\\ndef environment_snapshot(torch=None, device=None):\\n    versions = {}\\n    for name in (\\"torch\\", \\"transformers\\", \\"tokenizers\\", \\"safetensors\\", \\"numpy\\", \\"pandas\\", \\"scikit-learn\\", \\"iterative-stratification\\"):\\n        try:\\n            versions[name] = importlib.metadata.version(name)\\n        except importlib.metadata.PackageNotFoundError:\\n            versions[name] = None\\n    snapshot = {\\n        \\"python\\": platform.python_version(), \\"platform\\": platform.platform(),\\n        \\"machine\\": platform.machine(), \\"cpu_count\\": os.cpu_count(),\\n        \\"packages\\": versions, \\"device\\": str(device) if device is not None else None,\\n        \\"peak_process_rss_bytes\\": None,\\n        \\"measurement_scope\\": \\"PROCESS_ON_OBSERVED_ENVIRONMENT_NOT_AUTOMATIC_TARGET_VM_PROOF\\",\\n    }\\n    status = Path(\\"/proc/self/status\\")\\n    if status.exists():\\n        for line in status.read_text().splitlines():\\n            if line.startswith(\\"VmHWM:\\"):\\n                snapshot[\\"peak_process_rss_bytes\\"] = int(line.split()[1]) * 1024\\n    if torch is not None:\\n        snapshot[\\"torch_num_threads\\"] = torch.get_num_threads()\\n        snapshot[\\"cuda_available\\"] = torch.cuda.is_available()\\n        snapshot[\\"cuda_runtime\\"] = torch.version.cuda\\n        if device is not None and device.type == \\"cuda\\":\\n            snapshot[\\"gpu_name\\"] = torch.cuda.get_device_name(device)\\n            snapshot[\\"peak_cuda_allocated_bytes\\"] = torch.cuda.max_memory_allocated(device)\\n    return snapshot\\n\\n\\nclass SilarModelRunner:\\n    def __init__(self, sena_dir, nara_dir=None, *, device=\\"cpu\\", cpu_threads=None):\\n        self.model_evidence = {\\"sena\\": verify_model_directory(sena_dir, \\"sena\\")}\\n        if nara_dir is not None:\\n            self.model_evidence[\\"nara\\"] = verify_model_directory(nara_dir, \\"nara\\")\\n        try:\\n            import torch\\n            from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification\\n        except ImportError as error:\\n            raise RuntimeError(\\"PyTorch/Transformers belum siap; gunakan environment runtime pada petunjuk\\") from error\\n        if importlib.metadata.version(\\"transformers\\") != \\"5.13.1\\":\\n            raise RuntimeError(\\"Runner ini disiapkan untuk transformers==5.13.1 sesuai metadata aset\\")\\n        self.torch = torch\\n        self.device = torch.device(device)\\n        if self.device.type not in {\\"cpu\\", \\"cuda\\"}:\\n            raise ValueError(\\"Pilih cpu atau cuda secara eksplisit\\")\\n        if self.device.type == \\"cuda\\" and not torch.cuda.is_available():\\n            raise RuntimeError(\\"CUDA diminta tetapi tidak tersedia; tidak berpindah device diam-diam\\")\\n        if cpu_threads is not None:\\n            if type(cpu_threads) is not int or cpu_threads < 1:\\n                raise ValueError(\\"cpu_threads harus integer positif\\")\\n            torch.set_num_threads(cpu_threads)\\n        started = time.perf_counter()\\n        self.sena_tokenizer, self.sena_model = self._load(sena_dir, AutoTokenizer, AutoModelForSequenceClassification)\\n        self.nara_tokenizer = self.nara_model = None\\n        if nara_dir is not None:\\n            self.nara_tokenizer, self.nara_model = self._load(nara_dir, AutoTokenizer, AutoModelForTokenClassification)\\n        self.sync()\\n        self.load_seconds = time.perf_counter() - started\\n\\n    def _load(self, directory, tokenizer_class, model_class):\\n        path = str(Path(directory).resolve(strict=True))\\n        tokenizer = tokenizer_class.from_pretrained(path, local_files_only=True, trust_remote_code=False)\\n        model, loading = model_class.from_pretrained(\\n            path, local_files_only=True, trust_remote_code=False,\\n            use_safetensors=True, dtype=self.torch.float32, output_loading_info=True,\\n        )\\n        for key in (\\"missing_keys\\", \\"unexpected_keys\\", \\"mismatched_keys\\", \\"error_msgs\\"):\\n            if loading.get(key):\\n                raise RuntimeError(f\\"Pemuatan {model_class.__name__} tidak lengkap: {key}: {loading[key]}\\")\\n        model.to(self.device)\\n        model.eval()\\n        for parameter in model.parameters():\\n            parameter.requires_grad_(False)\\n        return tokenizer, model\\n\\n    def sync(self):\\n        if self.device.type == \\"cuda\\":\\n            self.torch.cuda.synchronize(self.device)\\n\\n    def predict_sena(self, text):\\n        torch = self.torch\\n        full_ids = self.sena_tokenizer(text, truncation=False, add_special_tokens=True)[\\"input_ids\\"]\\n        encoded = self.sena_tokenizer(text, return_tensors=\\"pt\\", truncation=True, max_length=MAX_LENGTH)\\n        inputs = {key: value.to(self.device) for key, value in encoded.items()}\\n        with torch.inference_mode():\\n            logits = self.sena_model(**inputs).logits[0]\\n            probabilities = torch.softmax(logits, dim=-1)\\n        pred_id = int(probabilities.argmax().item())\\n        labels = self.model_evidence[\\"sena\\"][\\"labels\\"]\\n        result = {\\n            \\"label\\": labels[pred_id], \\"confidence\\": float(probabilities[pred_id].item()),\\n            \\"probabilities\\": {labels[index]: float(probabilities[index].item()) for index in range(len(labels))},\\n        }\\n        validate_prediction(result)\\n        return result, {\\n            \\"source_wordpieces_with_special_tokens\\": len(full_ids),\\n            \\"processed_wordpieces_with_special_tokens\\": int(inputs[\\"input_ids\\"].shape[1]),\\n            \\"truncated\\": len(full_ids) > MAX_LENGTH,\\n        }\\n\\n    def predict_nara(self, text):\\n        if self.nara_model is None:\\n            raise RuntimeError(\\"NARA belum dimuat\\")\\n        torch = self.torch\\n        token_items = [{\\"token\\": match.group().lower(), \\"original\\": match.group(),\\n                        \\"start\\": match.start(), \\"end\\": match.end()} for match in TOKEN_PATTERN.finditer(text)]\\n        encoded = encode_nara_words([item[\\"token\\"] for item in token_items], self.nara_tokenizer)\\n        inputs = {\\n            key: torch.tensor([encoded[key]], dtype=torch.long, device=self.device)\\n            for key in (\\"input_ids\\", \\"attention_mask\\")\\n        }\\n        with torch.inference_mode():\\n            logits = self.nara_model(**inputs).logits[0]\\n            probabilities = torch.softmax(logits, dim=-1)\\n        labels, output = self.model_evidence[\\"nara\\"][\\"labels\\"], []\\n        for index, position in enumerate(encoded[\\"positions\\"]):\\n            pred_id = int(probabilities[position].argmax().item())\\n            output.append({**token_items[index], \\"label\\": labels[pred_id],\\n                           \\"confidence\\": float(probabilities[position, pred_id].item())})\\n        return output, {\\n            \\"source_regex_tokens\\": len(token_items), \\"processed_regex_tokens\\": len(output),\\n            \\"all_regex_tokens_processed\\": len(token_items) == len(output),\\n            \\"last_wordpiece_sequence_partial\\": encoded[\\"partial_last_word\\"],\\n            \\"processed_wordpieces_with_special_tokens\\": len(encoded[\\"input_ids\\"]),\\n            \\"alignment\\": \\"FIRST_WORDPIECE_LEGACY_INFERENCE_NOT_PROOF_OF_TRAINING_ALIGNMENT\\",\\n        }\\n\\n\\ndef run_inference(args):\\n    records = read_records(args.input)\\n    out = args.output_dir\\n    out.mkdir(parents=True, exist_ok=True)\\n    prediction_path, summary_path = out / \\"predictions.jsonl\\", out / \\"run_summary.json\\"\\n    if prediction_path.exists() or summary_path.exists():\\n        raise ValueError(\\"Folder hasil sudah berisi run; gunakan folder baru agar bukti sebelumnya terjaga\\")\\n    started = time.perf_counter()\\n    runner = None\\n    latencies = []\\n    count = 0\\n    summary = {\\n        \\"runner_version\\": RUNNER_VERSION, \\"adapter_version\\": ADAPTER_VERSION,\\n        \\"mode\\": args.command, \\"status\\": \\"RUNNING\\", \\"input_sha256\\": sha256_file(args.input),\\n        \\"model_inference_executed\\": False, \\"training_executed\\": False,\\n        \\"production_threshold_selected\\": False, \\"fusion_executed\\": False,\\n        \\"environment_label\\": args.environment_label,\\n        \\"environment\\": environment_snapshot(), \\"records_expected\\": len(records),\\n    }\\n    write_json(summary_path, summary)\\n    try:\\n        runner = SilarModelRunner(args.sena_dir, args.nara_dir if args.command == \\"smoke\\" else None,\\n                                  device=args.device, cpu_threads=args.cpu_threads)\\n        summary[\\"models\\"] = runner.model_evidence\\n        summary[\\"model_load_seconds\\"] = runner.load_seconds\\n        with prediction_path.open(\\"x\\", encoding=\\"utf-8\\") as stream:\\n            for record in records:\\n                runner.sync()\\n                before = time.perf_counter()\\n                event_prediction, sena_tokens = runner.predict_sena(record[\\"text\\"])\\n                # Prediksi neural pertama benar-benar sudah terjadi.\\n                summary[\\"model_inference_executed\\"] = True\\n                result = {\\"id\\": record[\\"id\\"], \\"input_sha256\\": text_digest(record[\\"text\\"]),\\n                          \\"event_classifier\\": event_prediction, \\"sena_token_coverage\\": sena_tokens}\\n                if args.command == \\"smoke\\":\\n                    nara_tokens, coverage = runner.predict_nara(record[\\"text\\"])\\n                    legacy_input, trace = build_adapter_input(record[\\"text\\"], event_prediction, nara_tokens)\\n                    result[\\"adapted_result\\"] = adapt_legacy_ai_output(\\n                        legacy_input, narrative_id=record[\\"id\\"],\\n                        sena_model_version=\\"SILAR_SENA_v1\\", nara_model_version=\\"SILAR_NARA_v1\\",\\n                    )\\n                    result[\\"nara_token_coverage\\"] = coverage\\n                    result[\\"postprocessing_trace\\"] = trace\\n                    result[\\"fusion_not_executed_reason\\"] = \\"PRODUCTION_THRESHOLD_NOT_CALIBRATED\\"\\n                runner.sync()\\n                result[\\"elapsed_seconds\\"] = time.perf_counter() - before\\n                latencies.append(result[\\"elapsed_seconds\\"])\\n                stream.write(json.dumps(result, ensure_ascii=False, allow_nan=False) + \\"\\\\n\\")\\n                stream.flush()\\n                count += 1\\n                if count == 1 or count % 25 == 0 or count == len(records):\\n                    print(f\\"{args.command}: {count}/{len(records)}\\", flush=True)\\n        summary[\\"status\\"] = \\"SUCCEEDED\\"\\n    except Exception as error:\\n        summary[\\"status\\"] = \\"FAILED\\"\\n        summary[\\"error\\"] = {\\"type\\": type(error).__name__, \\"message\\": str(error)}\\n        raise\\n    finally:\\n        summary[\\"records_completed\\"] = count\\n        summary[\\"wall_seconds_including_loading\\"] = time.perf_counter() - started\\n        if runner is not None:\\n            summary[\\"environment\\"] = environment_snapshot(runner.torch, runner.device)\\n        if latencies:\\n            summary[\\"per_record_seconds_including_first_record\\"] = {\\n                \\"min\\": min(latencies), \\"median\\": statistics.median(latencies),\\n                \\"mean\\": statistics.mean(latencies), \\"max\\": max(latencies),\\n                \\"scope\\": \\"SINGLE_PASS_NOT_STEADY_STATE_OR_CONCURRENT_BACKEND_BENCHMARK\\",\\n            }\\n        if prediction_path.exists():\\n            summary[\\"predictions_sha256\\"] = sha256_file(prediction_path)\\n        write_json(summary_path, summary)\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    commands = parser.add_subparsers(dest=\\"command\\", required=True)\\n    extract = commands.add_parser(\\"extract\\", help=\\"Verifikasi ZIP dan ekstrak berkas runtime yang dikunci\\")\\n    extract.add_argument(\\"--sena-zip\\", type=Path, required=True)\\n    extract.add_argument(\\"--nara-zip\\", type=Path, required=True)\\n    extract.add_argument(\\"--output-dir\\", type=Path, required=True)\\n    for command in (\\"smoke\\", \\"sena\\"):\\n        run = commands.add_parser(command)\\n        run.add_argument(\\"--sena-dir\\", type=Path, required=True)\\n        if command == \\"smoke\\":\\n            run.add_argument(\\"--nara-dir\\", type=Path, required=True)\\n        run.add_argument(\\"--input\\", type=Path, required=True)\\n        run.add_argument(\\"--output-dir\\", type=Path, required=True)\\n        run.add_argument(\\"--device\\", choices=(\\"cpu\\", \\"cuda\\"), required=True)\\n        run.add_argument(\\"--cpu-threads\\", type=int)\\n        run.add_argument(\\"--environment-label\\", required=True)\\n    args = parser.parse_args(argv)\\n    if args.command == \\"extract\\":\\n        evidence = {name: extract_model_archive(path, args.output_dir / name, name)\\n                    for name, path in ((\\"sena\\", args.sena_zip), (\\"nara\\", args.nara_zip))}\\n        write_json(args.output_dir / \\"extraction_evidence.json\\", evidence)\\n    else:\\n        run_inference(args)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "models/slr09/config/SLR09_Konfigurasi_Integrasi_v0.1.json": "{\\n  \\"schema\\": \\"slr09-integration-configuration-0.1\\",\\n  \\"status\\": \\"USER_SELECTED_FOR_INTEGRATION\\",\\n  \\"selected_option\\": \\"A\\",\\n  \\"confidence_threshold\\": 0.95,\\n  \\"calibration_reference\\": \\"SLR09_SENA_20261018T044447_926664Z:reviewed_validation620:cutoff_0.95\\",\\n  \\"decision_file\\": \\"config/threshold_decision.json\\",\\n  \\"low_confidence_rule\\": \\"confidence < confidence_threshold\\",\\n  \\"equal_to_threshold_is_low_confidence\\": false,\\n  \\"reference_run_id\\": \\"20261018T044447_926664Z\\",\\n  \\"reference_report_sha256\\": \\"0077365fac8b70d4c624958bbcef60978aab41cf50978bd78f8154c9bdbf691f\\",\\n  \\"reference_scope\\": \\"620_REVIEWED_VALIDATION_CHECKPOINT_SELECTION_NOT_INDEPENDENT_TEST\\",\\n  \\"probability_calibration_fitted\\": false,\\n  \\"selection_basis\\": \\"OBSERVED_VALIDATION_TRADEOFF_AND_EXPLICIT_USER_DECISION\\",\\n  \\"sena\\": {\\n    \\"archive_name\\": \\"SILAR_SENA_reviewed_pilot_20261018T044447_926664Z.zip\\",\\n    \\"archive_sha256\\": \\"144e13d1904f1f4beee56a5ac29fc4fc75cf6a424806ce3c9a0082bf6adf92a3\\",\\n    \\"weights_sha256\\": \\"9512a4895de281f3696e6c53a86d1271ad45180339995af01cf2b886ddc0afad\\",\\n    \\"run_id\\": \\"20261018T044447_926664Z\\",\\n    \\"labels\\": [\\n      \\"FLOOD\\",\\n      \\"EARTHQUAKE\\",\\n      \\"FIRE\\",\\n      \\"ACCIDENT\\",\\n      \\"NON_EVENT\\"\\n    ],\\n    \\"weight_identity_source\\": \\"COLAB_EXPORT_MANIFEST_NOT_DIRECT_WEIGHT_INSPECTION\\"\\n  },\\n  \\"nara\\": {\\n    \\"archive_name\\": \\"SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip\\",\\n    \\"archive_sha256\\": \\"bce9feba0dd2ae4518880708085f44286ae8d1dd7e50f4eb452e36824f967798\\",\\n    \\"weights_sha256\\": \\"ef84ccfeca2d116942c5d7f8882e064192279eabf92127e8adc1741933b60c29\\",\\n    \\"run_id\\": \\"20261018T001122_607337Z\\",\\n    \\"labels\\": [\\n      \\"O\\",\\n      \\"ACCIDENT-EVENT\\",\\n      \\"AFFECTEDOBJECTS-ARG\\",\\n      \\"DEATHVICTIM-ARG\\",\\n      \\"EARTHQUAKE-EVENT\\",\\n      \\"FALSE-EVENT\\",\\n      \\"FIRE-EVENT\\",\\n      \\"FLOOD-EVENT\\",\\n      \\"INFORMATION-ARG\\",\\n      \\"OFFICER-ARG\\",\\n      \\"PLACE-ARG\\",\\n      \\"REASON-ARG\\",\\n      \\"STREET-ARG\\",\\n      \\"TIME-ARG\\",\\n      \\"WOUNDVICTIM-ARG\\"\\n    ],\\n    \\"weight_identity_source\\": \\"COLAB_EXPORT_MANIFEST_NOT_DIRECT_WEIGHT_INSPECTION\\"\\n  },\\n  \\"adapter_version\\": \\"silar-adapter-0.4.0\\",\\n  \\"fusion_version\\": \\"silar-fusion-a4.22-0.1.0\\",\\n  \\"victim_guard_version\\": \\"silar-victim-guard-0.4.0\\",\\n  \\"role_guideline_version\\": \\"nara-role-guideline-0.2\\",\\n  \\"rules\\": {\\n    \\"category_from_sena_only\\": true,\\n    \\"non_event_blocks_submission\\": true,\\n    \\"human_verification_always_required\\": true,\\n    \\"empty_fields_do_not_automatically_make_uncertain\\": true,\\n    \\"unknown_extracted_values_remain_null_or_empty_array\\": true\\n  },\\n  \\"production_deployed\\": false,\\n  \\"production_model_selected\\": false,\\n  \\"quality_acceptance_approved\\": false,\\n  \\"target_vm_loading_and_joint_new_model_inference_verified\\": false,\\n  \\"general_extraction_conflict_detection_verified\\": false\\n}\\n", "models/slr09/config/candidate_asset_lock.json": "{\\n  \\"schema\\": \\"slr09-candidate-asset-lock-0.2\\",\\n  \\"source\\": \\"COLAB_EXPORT_MANIFESTS_NOT_A_NEW_DIRECT_WEIGHT_CHECK\\",\\n  \\"models\\": {\\n    \\"sena\\": {\\n      \\"model_id\\": \\"SILAR_SENA_reviewed_pilot_20261018T044447_926664Z\\",\\n      \\"run_id\\": \\"20261018T044447_926664Z\\",\\n      \\"archive_name\\": \\"SILAR_SENA_reviewed_pilot_20261018T044447_926664Z.zip\\",\\n      \\"archive_sha256\\": \\"144e13d1904f1f4beee56a5ac29fc4fc75cf6a424806ce3c9a0082bf6adf92a3\\",\\n      \\"labels\\": [\\n        \\"FLOOD\\",\\n        \\"EARTHQUAKE\\",\\n        \\"FIRE\\",\\n        \\"ACCIDENT\\",\\n        \\"NON_EVENT\\"\\n      ],\\n      \\"files\\": {\\n        \\"config.json\\": \\"53190f6443cea17db8e8d152839916b640aac910ff6e50c6bcb7f8cc4729e431\\",\\n        \\"model.safetensors\\": \\"9512a4895de281f3696e6c53a86d1271ad45180339995af01cf2b886ddc0afad\\",\\n        \\"tokenizer.json\\": \\"9fe770007d1727a8cacfec0b467e2b6d6f60d7ff93f3b0be692d63c62ccc856d\\",\\n        \\"tokenizer_config.json\\": \\"af1e05390d2f52f5c09553b0695dba1f047cadc875a0b12dca06c9e0fe039b21\\",\\n        \\"special_tokens_map.json\\": \\"5d5b662e421ea9fac075174bb0688ee0d9431699900b90662acd44b2a350503a\\",\\n        \\"vocab.txt\\": \\"343bf28a78f7efeda6419fa610f553184a6dcd29aae726f46850cd05e3167dc3\\"\\n      },\\n      \\"export_report\\": \\"sources/SLR09_SENA_Hasil_FineTune_20261018T044447_926664Z.zip\\",\\n      \\"export_report_sha256\\": \\"0077365fac8b70d4c624958bbcef60978aab41cf50978bd78f8154c9bdbf691f\\"\\n    },\\n    \\"nara\\": {\\n      \\"model_id\\": \\"SILAR_NARA_reviewed_pilot_20261018T001122_607337Z\\",\\n      \\"run_id\\": \\"20261018T001122_607337Z\\",\\n      \\"archive_name\\": \\"SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip\\",\\n      \\"archive_sha256\\": \\"bce9feba0dd2ae4518880708085f44286ae8d1dd7e50f4eb452e36824f967798\\",\\n      \\"labels\\": [\\n        \\"O\\",\\n        \\"ACCIDENT-EVENT\\",\\n        \\"AFFECTEDOBJECTS-ARG\\",\\n        \\"DEATHVICTIM-ARG\\",\\n        \\"EARTHQUAKE-EVENT\\",\\n        \\"FALSE-EVENT\\",\\n        \\"FIRE-EVENT\\",\\n        \\"FLOOD-EVENT\\",\\n        \\"INFORMATION-ARG\\",\\n        \\"OFFICER-ARG\\",\\n        \\"PLACE-ARG\\",\\n        \\"REASON-ARG\\",\\n        \\"STREET-ARG\\",\\n        \\"TIME-ARG\\",\\n        \\"WOUNDVICTIM-ARG\\"\\n      ],\\n      \\"files\\": {\\n        \\"config.json\\": \\"fa5336688409b42cb5e0baa58c039bb4ddc8c9253c4fc812625ac807c2e2d058\\",\\n        \\"model.safetensors\\": \\"ef84ccfeca2d116942c5d7f8882e064192279eabf92127e8adc1741933b60c29\\",\\n        \\"tokenizer.json\\": \\"9fe770007d1727a8cacfec0b467e2b6d6f60d7ff93f3b0be692d63c62ccc856d\\",\\n        \\"tokenizer_config.json\\": \\"4375ee5167df42507f6c3253ea0b6f65b04f7339ff8d6312754e37084e68ce9e\\",\\n        \\"special_tokens_map.json\\": \\"5d5b662e421ea9fac075174bb0688ee0d9431699900b90662acd44b2a350503a\\",\\n        \\"vocab.txt\\": \\"343bf28a78f7efeda6419fa610f553184a6dcd29aae726f46850cd05e3167dc3\\"\\n      },\\n      \\"export_report\\": \\"sources/SLR09_NARA_Hasil_FineTune_20261018T001122_607337Z.zip\\",\\n      \\"export_report_sha256\\": \\"879b3205419a9897075145a64c94e29c1f3b81b558af170a8181f55fdfa63bfb\\"\\n    }\\n  }\\n}\\n", "models/slr09/config/inference_environment.json": "{\\n  \\"schema\\": \\"slr09-inference-environment-0.2\\",\\n  \\"profile_id\\": \\"CANDIDATE_EXPORT_STACK_PENDING_LOCAL_SMOKE\\",\\n  \\"basis\\": \\"VERSIONS_RECORDED_IN_COMPLETED_COLAB_RUNS\\",\\n  \\"python_recommended\\": \\"3.12 or 3.13\\",\\n  \\"pinned_packages\\": {\\n    \\"torch\\": \\"2.11.0\\",\\n    \\"transformers\\": \\"4.57.1\\",\\n    \\"tokenizers\\": \\"0.22.1\\",\\n    \\"safetensors\\": \\"0.8.0\\",\\n    \\"numpy\\": \\"2.1.3\\"\\n  },\\n  \\"guard_packages\\": [\\n    \\"torch\\",\\n    \\"transformers\\",\\n    \\"tokenizers\\",\\n    \\"safetensors\\",\\n    \\"numpy\\"\\n  ],\\n  \\"candidate_neural_loading_verified_in_this_revision\\": false,\\n  \\"scope\\": \\"DEPENDENCY_PROFILE_PREPARED_NOT_PROOF_OF_TARGET_ENVIRONMENT\\"\\n}\\n", "models/slr09/config/requirements-inference.txt": "# Profil dari versi Colab yang tercatat; smoke test kandidat di lingkungan tujuan masih diperlukan.\\n# Untuk CPU/CUDA, gunakan build PyTorch yang sesuai; local suffix dicatat pada log.\\ntorch==2.11.0\\ntransformers==4.57.1\\ntokenizers==0.22.1\\nsafetensors==0.8.0\\nnumpy==2.1.3\\n", "models/slr09/examples/smoke_inputs.jsonl": "{\\"id\\": \\"SMOKE-SYNTHETIC-001\\", \\"text\\": \\"Banjir merendam dua rumah di Rajabasa tadi pagi.\\", \\"source\\": \\"SYNTHETIC_INPUT_FOR_LOADING_SMOKE_NOT_QUALITY_TEST\\"}\\n{\\"id\\": \\"SMOKE-SYNTHETIC-002\\", \\"text\\": \\"Petugas melakukan simulasi kebakaran di sekolah.\\", \\"source\\": \\"SYNTHETIC_INPUT_FOR_LOADING_SMOKE_NOT_QUALITY_TEST\\"}\\n{\\"id\\": \\"SMOKE-SYNTHETIC-003\\", \\"text\\": \\"Kecelakaan di Jalan Mawar menyebabkan satu warga terluka.\\", \\"source\\": \\"SYNTHETIC_INPUT_FOR_LOADING_SMOKE_NOT_QUALITY_TEST\\"}\\n"}')
EXPECTED_BUNDLE_SHA256 = 'd59e95a26493bbe005593cc67af478bd0b7b963557849cb2427669f167a8be09'
actual_bundle = hashlib.sha256(json.dumps(EMBEDDED_FILES, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
assert actual_bundle == EXPECTED_BUNDLE_SHA256, "Snapshot kode tidak cocok dengan identitas notebook"

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
WORK_ROOT = Path(os.environ.get("SLR09_WORK_ROOT", "/content")) / ("slr09_candidate_v02_" + RUN_ID)
WORK_ROOT.mkdir(parents=True, exist_ok=False)
for name, text in EMBEDDED_FILES.items():
    path = WORK_ROOT / name
    assert path.resolve().is_relative_to(WORK_ROOT.resolve())
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(text.encode("utf-8"))

PACKAGE_ROOT = WORK_ROOT / "models/slr09"
PROFILE = json.loads((PACKAGE_ROOT / "config/inference_environment.json").read_text())
CONFIG = json.loads((PACKAGE_ROOT / "config/SLR09_Konfigurasi_Integrasi_v0.1.json").read_text())
RESULTS = WORK_ROOT / "evidence"
RESULTS.mkdir()
SOURCE_HASHES = {name: hashlib.sha256(text.encode()).hexdigest() for name, text in EMBEDDED_FILES.items()}
(RESULTS / "source_snapshot.json").write_text(json.dumps({"source_package_sha256": "4d004d3e3f74cb8d698c67ee5571f03653b568e52455e28d025d1ff3dd78e2d6", "bundle_sha256": EXPECTED_BUNDLE_SHA256, "files_sha256": SOURCE_HASHES}, indent=2) + "\n")
print("Snapshot kode siap:", len(EMBEDDED_FILES), "berkas; run", RUN_ID)
print("Cutoff terpilih:", CONFIG["confidence_threshold"])


## 2. Periksa dependency inference

Hanya dependency yang tidak cocok yang memicu instalasi profil. Versi mengikuti profil kandidat dari hasil eksperimen sebelumnya. Build PyTorch dengan suffix CPU/CUDA diterima jika versi pokok cocok.

Jika sel meminta **Restart session**, lakukan restart lalu jalankan notebook dari awal agar modul lama tidak tertinggal di memori. Ini restart kernel, bukan pelatihan ulang. Profil belum dianggap tervalidasi sampai kedua bobot benar-benar berhasil dijalankan.


In [ ]:
# 2. Siapkan dependency sebelum memuat model atau menghubungkan Drive.
expected = PROFILE["pinned_packages"]
observed = {}
for name in PROFILE["guard_packages"]:
    try:
        observed[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        observed[name] = None
different = [name for name in PROFILE["guard_packages"] if not observed[name] or observed[name].split("+", 1)[0] != expected[name]]
already_loaded = [name for name in different if name in sys.modules]
print("Versi sebelum pemeriksaan:", json.dumps(observed, indent=2))
if different:
    command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(PACKAGE_ROOT / "config/requirements-inference.txt")]
    completed = subprocess.run(command, capture_output=True, text=True)
    log = completed.stdout + completed.stderr
    (RESULTS / "dependency_install.txt").write_text(log)
    print(log[-6000:])
    if completed.returncode:
        raise RuntimeError("Dependency belum dapat dipasang; simpan error. Tidak menjalankan model atau training.")
    importlib.invalidate_caches()
    if already_loaded:
        raise RuntimeError("Restart session Colab, lalu jalankan notebook dari awal. Modul lama sudah ada di memori: " + ", ".join(already_loaded))
else:
    (RESULTS / "dependency_install.txt").write_text("Tidak ada instalasi: versi metadata telah sesuai profil.\n")
sys.path.insert(0, str(PACKAGE_ROOT / "code/active_nara_adapter"))
from slr09_candidate_runtime import check_dependencies, selected_assets
versions = check_dependencies(PROFILE)
_, LOCK, _ = selected_assets(PACKAGE_ROOT)
(RESULTS / "dependency_versions.json").write_text(json.dumps(versions, indent=2) + "\n")
print("Dependency sesuai profil. Pemuatan bobot belum dijalankan pada sel ini.")


## 3. Hubungkan Drive dan periksa lokasi ekspor

Nama berikut mengikuti manifest yang sudah ada. Jika arsip dipindahkan, ubah `EXPORT_DIR` ke folder yang benar; identitas hash tetap diperiksa. Notebook tidak menimpa atau mengubah kedua arsip.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
EXPORT_DIR = Path("/content/drive/MyDrive/SILAR_SLR09")
SENA_ZIP = EXPORT_DIR / CONFIG["sena"]["archive_name"]
NARA_ZIP = EXPORT_DIR / CONFIG["nara"]["archive_name"]
for name, path in [("SENA", SENA_ZIP), ("NARA", NARA_ZIP)]:
    print(name, path.name, "TERSEDIA" if path.is_file() else "BELUM DITEMUKAN")
if not SENA_ZIP.is_file() or not NARA_ZIP.is_file():
    raise FileNotFoundError("Arsip ekspor belum ditemukan. Perbaiki EXPORT_DIR; tidak perlu menjalankan training lagi.")


## 4. Jalankan satu uji singkat: hash → muat dua model → adapter → Fusion

Default menggunakan **CPU**. Tiga input sintetis hanya memeriksa jalur eksekusi; keluaran tidak dibandingkan dengan gold untuk membuat klaim accuracy baru. Cutoff tetap 0,95. Tidak menjalankan endpoint Django, submit, atau publikasi.

Sel menghasilkan ZIP bukti berukuran kecil tanpa bobot model. ZIP disalin ke folder ekspor Drive dan dapat diunduh. Jika proses gagal, hasil tetap ditandai **FAILED** beserta tahap/error, tanpa mengklaim pengujian berhasil.


In [ ]:
# 4. Pemeriksaan aktual melalui loader v0.2; tidak ada training.
from types import SimpleNamespace
import shutil
import traceback
import zipfile
from slr09_candidate_runtime import extract_candidate_archive, run_smoke

DEVICE = "cpu"  # Ganti menjadi "cuda" hanya bila GPU Colab tersedia.
phase = "EXTRACT_AND_VERIFY_ARCHIVES"
workflow = {"schema": "slr09-candidate-v02-smoke-workflow-0.1", "run_id": RUN_ID,
            "status": "RUNNING", "training_executed": False,
            "environment_scope": "REPRESENTATIVE", "quality_acceptance_approved": False,
            "target_vm_verified": False, "source_bundle_sha256": EXPECTED_BUNDLE_SHA256}
failure = None
archive_path = None
try:
    extracted = {}
    for name, source in [("sena", SENA_ZIP), ("nara", NARA_ZIP)]:
        print("Memeriksa arsip dan hash:", name.upper(), flush=True)
        extracted[name] = extract_candidate_archive(source, WORK_ROOT / "runtime_models" / name, name, LOCK["models"][name])
    (RESULTS / "extraction_evidence.json").write_text(json.dumps(extracted, ensure_ascii=False, indent=2) + "\n")
    phase = "LOAD_AND_INFER_BOTH_CANDIDATES"
    args = SimpleNamespace(root=PACKAGE_ROOT, sena_dir=WORK_ROOT / "runtime_models/sena",
        nara_dir=WORK_ROOT / "runtime_models/nara", input=PACKAGE_ROOT / "examples/smoke_inputs.jsonl",
        output_dir=RESULTS / "smoke", device=DEVICE, cpu_threads=2 if DEVICE == "cpu" else None,
        environment_label="GOOGLE_COLAB_" + DEVICE.upper(), environment_scope="REPRESENTATIVE")
    print("Memuat kedua kandidat dan menjalankan tiga input melalui adapter/Fusion...", flush=True)
    summary = run_smoke(args)
    phase = "VERIFY_RESULT_IDENTITY_AND_CONTRACT"
    predictions = [json.loads(line) for line in (args.output_dir / "predictions.jsonl").read_text().splitlines() if line.strip()]
    assert summary["status"] == "SUCCEEDED"
    assert summary["records_completed"] == summary["records_expected"] == len(predictions) == 3
    assert all(summary[name] for name in ["model_inference_executed", "sena_inference_executed", "nara_inference_executed", "fusion_executed"])
    assert summary["confidence_threshold"] == .95
    for name in ["sena", "nara"]:
        assert summary["models"][name]["model_id"] == LOCK["models"][name]["model_id"]
        assert summary["models"][name]["files"]["model.safetensors"] == CONFIG[name]["weights_sha256"]
        assert extracted[name]["archive_sha256_observed"] == CONFIG[name]["archive_sha256"]
    extraction_fields = {"location_text", "time_text", "occurred_at", "involved_parties", "impacts", "victims", "affected_objects"}
    for row in predictions:
        adapted, fused = row["adapted_result"], row["fusion_result"]
        assert set(adapted["nara_result"]["normalized_output"]) == extraction_fields
        assert set(fused["incident_card_version_fields"]["data"]) == extraction_fields | {"event_type"}
        for name in ["sena", "nara"]:
            assert LOCK["models"][name]["model_id"] in adapted[name + "_result"]["model_version"]
        category = row["event_classifier"]["label"]
        assert fused["integration_signals"]["blocked_by_non_event"] is (category == "NON_EVENT")
        assert fused["incident_card_version_fields"]["data"]["event_type"] == (None if category == "NON_EVENT" else category)
    workflow.update(status="SUCCEEDED", phase="COMPLETE", records_completed=len(predictions),
                    model_inference_executed=True, fusion_executed=True)
    print("BERHASIL: kedua kandidat, identitas bobot dan jalur loader/adapter/Fusion sesuai.")
except Exception as error:
    failure = error
    workflow.update(status="FAILED", phase=phase, error={"type": type(error).__name__, "message": str(error)})
    summary_path = RESULTS / "smoke/run_summary.json"
    workflow["model_inference_executed"] = json.loads(summary_path.read_text()).get("model_inference_executed", False) if summary_path.exists() else False
    (RESULTS / "error_trace.txt").write_text(traceback.format_exc())
    print("BELUM LULUS pada tahap", phase, ":", error)
finally:
    (RESULTS / "workflow_result.json").write_text(json.dumps(workflow, ensure_ascii=False, indent=2) + "\n")
    for name in ["SLR09_Konfigurasi_Integrasi_v0.1.json", "candidate_asset_lock.json", "inference_environment.json"]:
        shutil.copy2(PACKAGE_ROOT / "config" / name, RESULTS / name)
    shutil.copy2(PACKAGE_ROOT / "examples/smoke_inputs.jsonl", RESULTS / "smoke_inputs.jsonl")
    (RESULTS / "README_hasil.txt").write_text("Bukti uji loader kandidat v0.2 di Google Colab (REPRESENTATIVE).\nTidak ada training, evaluasi mutu independen atau deployment.\nStatus kegiatan dibaca dari workflow_result.json; jangan menganggap ZIP bukti selalu berarti PASS.\n")
    archive_path = WORK_ROOT / ("SLR09_Hasil_Smoke_Kandidat_v02_" + RUN_ID + ".zip")
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as output:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                output.write(path, str(path.relative_to(RESULTS)))
    saved_path = EXPORT_DIR / archive_path.name
    try:
        shutil.copy2(archive_path, saved_path)
        print("ZIP bukti tersimpan:", saved_path)
    except OSError as save_error:
        print("Penyimpanan ke Drive gagal:", save_error, "; gunakan unduhan dari /content.")
    print("Status akhir:", workflow["status"])
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception as download_error:
        print("Unduh manual dari panel Files:", archive_path, ";", download_error)
if failure is not None:
    raise RuntimeError("Uji kandidat belum lulus; kirim ZIP bukti yang dihasilkan untuk diperiksa.") from failure


## Setelah notebook selesai

Kirim **ZIP hasil smoke** yang dicetak pada sel terakhir. File tersebut memuat hash, identitas kandidat, lingkungan, hasil tiga input, tujuh field adapter dan kartu Fusion serta status keberhasilan/kegagalan. Tidak perlu mengirim bobot kembali untuk membaca hasil ini.

Jika lulus, langkah berikutnya adalah menilai dasar penerimaan mutu dan melengkapi keputusan kelompok pada `review-results.md`. Bukti evaluasi sebelumnya tetap dipakai sesuai cakupannya; SENA 620 tetap validation pemilihan checkpoint, NARA 79 tetap subset revisi. Tentukan pengujian kualitas tambahan berdasarkan kekurangan yang nyata, bukan mengulang training otomatis.

Uji di VM tujuan atau persetujuan bahwa Colab mewakili lingkungan yang dipilih tetap perlu dicatat sesuai card. Kelulusan notebook tidak otomatis memindahkan SLR_09 menjadi Done dan tidak menetapkan model produksi.
